# The bee's knees: reading MRI with an anatomical mirror, and a second reader

![An artistic knee joint with mirrored MRI planes](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/knee-anatomical-mirror-hero-v1.png)

*AI-generated artistic illustration, not measured MRI. [Prompt and artwork provenance](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/ARTWORK_PROVENANCE.json).*

A knee MRI is a collection of acquisitions, not one picture. This notebook turns that collection into a finding vector in two ways, and blends them. The first arm reads adjacent slice triplets with a public checkpoint and averages a plain and an anatomically mirrored view. The second arm is a released three-fold ConvNeXt reader from another team. The blend is a rank average at one predeclared weight, 0.70 for the mirror arm and 0.30 for the ConvNeXt arm. The implementation is readable enough to fork.

## Results so far

| Entry | Score | Basis |
|---|---|---|
| nartaa V2: upstream recipe and checkpoint | 0.949 | Their score on their notebook. Attributed here, not reproduced |
| This public notebook (card) | 0.941 | Official rows 56759401 (V31), 56779708 and 56779710 (V32), all 0.941 |
| Account's verified best | 0.943 | Private notebook, row 56859861 (V8) |
| Public frontier, observed 2026-10-08 | Six notebooks at 0.950 (blends) | Public leaderboard. Not ours |
| Private validation bench, V14 (mirror arm) | 0.949 | Reported by the maintainer; not part of this public notebook |
| Private validation bench, V17 (E3 r224 blend) | 0.949 | Reported by the maintainer; ties V14 |
| Private validation bench, V18 (ConvNeXt arm, decoder fix) | no score yet | Arm ran on all 3 visible studies; oracle parity max abs diff 2.98e-08 against tolerance 2e-4 |
| This notebook, version 34 | 0.949 | Official row 56949940, complete |
| This notebook, version 35 | pending | Adds the ConvNeXt arm. No score is claimed until a completed official submission exists |

Earlier versions appear in the preserved history below, with their cells unchanged. Version 35 changes the active path, so its own native run and official submission are still to come.

## What changed in version 35

The clock starts in the first statement of the notebook, before any import, staging or CUDA work, so the time budget covers setup as well as inference. Both arms run inside the same owned process-group watchdog, which cleans up the whole process tree on success as well as on failure. The mirror arm is validated before anything else. The ConvNeXt arm runs only if the remaining budget clears a time gate. Each decoder wheel is installed on its own, best effort. The blend is written, and only then does the notebook export `/kaggle/working/submission.csv`, exclusively and last. If the ConvNeXt arm fails for any reason, the file is the mirror arm alone, and the receipt says so.

The blend in this notebook is predeclared: 0.70 for the mirror arm (nartaa's recipe and checkpoints, Apache-2.0) and 0.30 for goodpjw2008's ConvNeXt 2.5D reader (Apache-2.0). We have not reproduced the public 0.950 notebooks and make no claim about their internals. Credits are in the notices cell.


## Read the experiment before running it

![MRI volume, windows, two views, attention and ranks](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/anatomical_mirror_pipeline_v2.png)

| Choice | This baseline | Why it matters |
|---|---|---|
| Mirror arm: checkpoint | One selected public SWA checkpoint, exact V1 hash | Reproducible identity; this is not a new independent model |
| Mirror arm: field of view | 140 mm center crop | Pixels alone do not fix the anatomical field of view |
| Mirror arm: input | 94 adjacent triplets, center crop to 320 × 320 | Keep neighboring-slice information without resizing again |
| Mirror arm: mirror | Acquisition channels for sagittal centers; image width otherwise | A reflection must follow the acquisition plane |
| ConvNeXt arm: reader | goodpjw2008's three-fold ConvNeXt 2.5D reader, released sources, DICOM preprocessing as published | A second, independently trained reader with its own preprocessing |
| Blend | 0.70 × rank(mirror) + 0.30 × rank(ConvNeXt), then the rank of that score | One global weight, fixed before any run; no per-finding weights |
| Time gate | ConvNeXt runs only if the remaining budget clears 875 s (commit) or 8,100 s (rerun) | A slow arm can never delay the mirror-only fallback |

The baseline needs one or more **T4 GPUs** (the qualified environment used two), the publisher's original weights dataset V1, the goodpjw ConvNeXt dataset V1, and competition inputs you are authorized to access. The qualified container has Torch `2.11.0+cu128` and timm `1.0.29`; library checks fail before any model is loaded. Set `MODE="canary"` for a three-study smoke test of the mirror arm; it writes a separate file and is not an accuracy measurement. A notebook run only creates predictions. It never submits to the competition.


## 1. A small configuration you can actually use

The first code cell starts the clock with its first statement, before any import or staging, so the budget includes setup. A competition rerun (`KAGGLE_IS_COMPETITION_RERUN` set to true or 1) uses the published 32,400-second GPU limit. A commit uses the 1,800-second notebook limit. Each budget keeps a 50-second margin, so the whole-run deadlines are 32,350 s and 1,750 s. An unknown flag value stops the run.

The ConvNeXt time gate threshold is the smaller of the full estimate plus margin (7,200 s + 900 s) and half the whole budget. On a commit that is 875 s; on a rerun it is 8,100 s. Inside the arm, the child process gets the whole budget minus a 120-second reserve, so the blend and export always keep their time. Change `MODE`, `PREP_WORKERS` or the gate constants for a bounded experiment; paths change only if your attached data layout differs. If you change the geometry, the reader or the blend, update the pins and document the change before treating a fork as reproducible.


In [ ]:
_T0 = __import__("time").monotonic()  # the owned-run clock: the first statement of this notebook
import datetime, hashlib, json, os, sys, time
from pathlib import Path

CONFIG = {
    "MODE": "inference",                 # "inference": all test rows, the arms, the root submission; "canary": mirror arm on three studies, no export
    "PREP_WORKERS": 2,                   # the qualified value; supported 1, 2, 4
    "MAX_CANARY_STUDIES": 3,
    "INPUT_ROOT": Path("/kaggle/input"),
    "COMPETITION_ROOT": None,            # None: detect the mounted competition
    "OUTPUT_ROOT": Path("/kaggle/working"),
    "NOTEBOOK_LIMIT_SECONDS": 1800,      # the commit runtime limit
    "COMPETITION_LIMIT_SECONDS": 32400,  # the published nine-hour GPU limit for competition reruns
    "SAFETY_MARGIN_SECONDS": 50,
    "EXPECTED_TORCH": "2.11.0+cu128",
    "EXPECTED_TIMM": "1.0.29",
    "CONVNEXT_DEVICE": "cuda",
    "CONVNEXT_WORKERS": 4,
    "CONVNEXT_ESTIMATE_S": 7200,         # conservative: two hours for about 1,300 studies
    "CONVNEXT_GATE_MARGIN_S": 900,       # fifteen minutes of headroom
    "CONVNEXT_RESERVE_S": 120,           # kept for the blend and the root export
    "MIN_CONVNEXT_OK_FRACTION": 0.5,
    "MIRROR_WEIGHT": 0.70,               # predeclared, one global weight
    "CONVNEXT_WEIGHT": 0.30,
}
READER_SHA256 = "e9ec648e1b2f6121f36bbe6ae806a5714571db83804c719ea02d5c4bf7fe47d7"
OWNED_WATCHDOG_SHA256 = "54636411f9f20aeacdeec784b3cb4b5e5a3fbb8c6ea2ec334f0d0d41b6e6130e"
CHILD_SHA256 = "562d97e57fccfb4c132b2559fb960bac92e398140788a600ccd0ee1a1c344e2e"
CHECKPOINT_SHA256 = "7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef"
VISIBLE_MIRROR_SHA256 = "faa20e95a1779658237252388407eaf6d835679dd43e4dae5bca18d95496bede"
CONVNEXT_DATASET = "goodpjw2008/rsna-knee-2-5d-convnext-reader"
CHECKPOINT_NAMES = ["cnxt_v0_fold%d.pt" % _i for _i in range(3)]
PIP_COMMAND = [sys.executable, "-m", "pip"]
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion",
          "Synovitis", "Baker's", "Contusion", "Fracture"]
ASSET_PINS = {'cnxt_v0_fold0.pt': {'bytes': 123035271, 'sha256': '0b64baa9d1c29c8d5c4dffe06d9ab2c941119b37738e5a2346e882ab694e7b80'}, 'cnxt_v0_fold1.pt': {'bytes': 123035271, 'sha256': '61696b6e67474695242fc8049e96e2ea13987db6e20d04410dc2961a689585cb'}, 'cnxt_v0_fold2.pt': {'bytes': 123035271, 'sha256': 'f5cf19cf591600c1c1a04f1aa5ff467751ef01513426a4ea3baa789443656cb6'}, 'pylibjpeg-2.1.0-py3-none-any.whl': {'bytes': 25472, 'sha256': '25df9496a69e64e98c887fddee12a1271e275b5f74ba804f9bf98a08bb80993e'}, 'pylibjpeg_libjpeg-2.4.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'bytes': 3409132, 'sha256': '01d950ef496476a9223e4966376cb88098fcf5c55a12a21f722d7b5f84daae43'}, 'pylibjpeg_openjpeg-2.6.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'bytes': 1997697, 'sha256': 'aa6b08327be6f6daf0814a61fd7410f53a08e8e0b35774c634d9907402ff4f1c'}}
DECODER_NAMES = ['pylibjpeg-2.1.0-py3-none-any.whl', 'pylibjpeg_libjpeg-2.4.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl', 'pylibjpeg_openjpeg-2.6.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl']
RELEASED_SOURCE_PINS = {'preprocess.py': '4491963e4d2156625c9cf79051f4a25c32d2d761f1f07df3117a439ed8029ae2', 'knee.py': 'a24e703cb6446bc44922d6e4b9db40045e5047970c1a6272a1d295bd3783a412', 'infer.py': 'ce1a55d9b837d159d882e04721027d8a2c55af15d49f4ae66689fe3bbf8e5066'}
ORACLE = {'source': 'goodpjw2008 notebook V2 / script355300588 / _own.csv', 'sha256': '1606432b738e10e2ed05223ca67ecb3dd6e92195469f9fbb5cd6aa80ca17f55f', 'study_ids': ['1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191', '1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776', '1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402'], 'labels': ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'], 'probabilities': [[0.05028264, 0.13892183, 0.8572949, 0.0869992, 0.81512034, 0.21660852, 0.26661432, 0.5246947, 0.16817574, 0.96004313, 0.09890376, 0.06294121], [0.49914175, 0.20313464, 0.7575063, 0.07240992, 0.5393879, 0.29069966, 0.5636684, 0.30499026, 0.35689425, 0.25135294, 0.4342749, 0.19144803], [0.027257841, 0.04780493, 0.22868463, 0.199918, 0.74611175, 0.50843084, 0.94428635, 0.35745567, 0.15767214, 0.113095105, 0.09559411, 0.09150316]], 'max_abs_tolerance': 0.0002}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


_RERUN_RAW = os.environ.get("KAGGLE_IS_COMPETITION_RERUN")
_RERUN_NORMALIZED = None if _RERUN_RAW is None else _RERUN_RAW.lower()
require(_RERUN_NORMALIZED in (None, "", "false", "0", "true", "1"), "Unknown competition-rerun flag; no budget fallback")
COMPETITION_RERUN = _RERUN_NORMALIZED in ("true", "1")
SELECTED_LIMIT = CONFIG["COMPETITION_LIMIT_SECONDS"] if COMPETITION_RERUN else CONFIG["NOTEBOOK_LIMIT_SECONDS"]
WHOLE_BUDGET = SELECTED_LIMIT - CONFIG["SAFETY_MARGIN_SECONDS"]
FULL_DEADLINE = _T0 + WHOLE_BUDGET
# Gate threshold: the full estimate plus margin, but never more than half the whole budget (commit: 875 s).
CONVNEXT_THRESHOLD_S = min(CONFIG["CONVNEXT_ESTIMATE_S"] + CONFIG["CONVNEXT_GATE_MARGIN_S"], 0.5 * WHOLE_BUDGET)
RUN_TAG = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SOURCE_DIR = CONFIG["OUTPUT_ROOT"] / "public_v35_source"
SOURCE_DIR.mkdir(parents=True, exist_ok=False)
READER_PATH = SOURCE_DIR / "public_mirror_reader.py"
WATCHDOG_PATH = SOURCE_DIR / "owned_watchdog.py"
CHILD_PATH = SOURCE_DIR / "convnext_child.py"
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1")
print({"budget_seconds": WHOLE_BUDGET, "convnext_gate_threshold_s": CONVNEXT_THRESHOLD_S,
       "competition_rerun": COMPETITION_RERUN, "mode": CONFIG["MODE"], "competition_submission": False, "official_score": None})


## 2. Respect the acquisition geometry

| Slot | Source slice indices | Count |
|---|---|---:|
| Sagittal, fluid-sensitive | 0–25 | 26 |
| Sagittal, non-fluid-sensitive | 26–47 | 22 |
| Coronal, fluid-sensitive | 48–65 | 18 |
| Coronal, non-fluid-sensitive | 66–77 | 12 |
| Axial, any sequence | 78–95 | 18 |

We prefer the first suitable unused series in metadata order, falling back within the requested plane. DICOM position and orientation order the slices, with `InstanceNumber` as a fallback. Apply the modality LUT and MONOCHROME1 inversion before robust per-series intensity scaling. Sample the 2–98% index span, use the selected images' 2nd/98th pixel percentiles, crop a 140 mm field of view, then area-resize and quantize to uint8.

The volume layout survives missing acquisition slots. A study with no usable acquisition fails instead of receiving model priors. Window selection preserves the upstream span and can cross slot boundaries; the center determines the mirror operation.

![Acquisition-channel reversal and width reflection before normalization](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/mirror_before_normalization_v2.png)

Mirror the **uint8** triplet first. Then crop the central 320 × 320 region (32 pixels from each edge of 384 × 384), and only then apply the three channel-specific ImageNet means and standard deviations. Reversing channels after normalization would alter the recipe because those statistics differ.


## 3. The reader, and why tolerance matters

The next cell writes the exact version-34 reader (sha256 `e9ec648e…`). The launcher verifies those bytes before it starts. The same tolerant reader, its diff from the strict version, the offline test harness and the visible-run receipts are published in the [official-results folder](https://github.com/cubres/rsna-knee-abnormality-detection/tree/b8e1d6e89c391e27821a41455d5f42336f70420d/official-results-2026-10-08) of the [repository](https://github.com/cubres/rsna-knee-abnormality-detection). The architecture and the happy-path arithmetic are the same as before. What changed is how the reader treats bad input.

An earlier private version of this pipeline ran the same architecture under a strict reader. That reader stopped the whole job on an unreadable DICOM header, an unreadable or non-2D pixel array, a study with no usable slice, any per-study model error, or a device count other than two. Its visible three-study run passed. Its rerun on the hidden test cohort, about 1,300 studies, ended with an unhandled error and no score. We cannot see the hidden log, so this is our most likely explanation rather than a proven one. A cohort that large is very likely to contain at least one file that a strict reader rejects, and one such file should not zero the submission.

The public reference handles these cases deliberately. An unreadable header keeps its file as a neutral record. An unreadable slice is skipped and its slot stays zero. A study whose volume cannot be built is predicted from a zero volume and mask. A failed model call gives that study 0.5 probabilities. Every study still receives a prediction. In offline tests with synthetic DICOM, this reader matched the reference on truncated, garbage, multi-frame, NaN and missing-series studies. Its output was identical to the strict reader's on healthy studies. Those tests do not measure accuracy, and the GPU path still needs its own native run.

The general lesson is that a competition reader is part of the scoring pipeline. Its failure policy should be stated, tested on bad inputs, and chosen so that every study stays in the output. The functions to read are `build_volume` (the fault policy), `prepare_windows` and `predict_pair` (the model path), and `run` (coverage and ranks).


In [ ]:
%%writefile /kaggle/working/public_v35_source/public_mirror_reader.py
"""Original inference implementation of the qualified public Knee reader.

Architecture and geometry: dreaddevelopment's Raptor CoAtNet/MIL lineage.
Selected learned weights and plain/mirror recipe: nartaa public V2,
https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397
Notebook source is Apache-2.0. Weights have separate research/educational terms,
including this competition. This module grants no rights to OAI source data.

Fault policy (V14): the same tolerant semantics as the public reference's
reader (its header record, per-slice pixel skip and whole-study fallback).
A header failure keeps the file as a (0.0, path, 0.5) record; a pixel failure
leaves that slot zero and advances the slot index; a study whose volume cannot
be built is predicted from a zero volume and zero mask; a model failure gives
that study 0.5 probabilities. No study is dropped, and any CUDA device count
of at least one is accepted.

Importing this file performs no model load, network request or native inference.
Only main() starts an explicitly requested local Kaggle experiment.
"""
from __future__ import annotations

import argparse
from concurrent.futures import ThreadPoolExecutor
import glob
import hashlib
import json
import os
from pathlib import Path
import queue
import threading
import time

PUBLIC_REF = "nartaa/rsna-knee-0949-anatomical-mirror"
PUBLIC_VERSION = 2
PUBLIC_SESSION_ID = 356175397
PUBLIC_NOTEBOOK_SHA256 = "8f1f96530b08543bd6caeed48489eea7c01657d6994f22e21a8400047fd4e100"
PUBLIC_INFERENCE_CELL_SHA256 = "bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5"
DATASET_REF = "nartaa/rsna-knee-publication-swa-weights-20261007"
DATASET_VERSION = 1
CHECKPOINT_NAME = "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt"
CHECKPOINT_SHA256 = "7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef"
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
LABELS = ("ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture")
SLOTS = (("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18))
CORPUS_RES = 384
INPUT_RES = 320
SLICE_COUNT = 96
WINDOW_COUNT = 94
CROP_MM = 140.0
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)
FAULT_POLICY = ("V14 reference-tolerant: header failure keeps (0.0, 0.5) record; pixel failure skips slice "
                "(zero slot, index advances); study build failure gives zero volume and mask; model failure "
                "gives 0.5 probabilities; no study is dropped")
PROBABILITY_DEFAULT = 0.5


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def mirror_axis_for_center(center):
    require(type(center) is int and 0 <= center < SLICE_COUNT, "Invalid centre slice")
    return "channels" if center < 48 else "width"


def find_attached_checkpoint(root):
    candidates = (
        root / DATASET_REF.split("/")[1] / CHECKPOINT_NAME,
        root / "datasets" / DATASET_REF / CHECKPOINT_NAME,
    )
    available = [path for path in candidates if path.is_file()]
    require(bool(available), "Attach the qualified public dataset V1; checkpoint is absent")
    path = available[0]
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    require(digest == CHECKPOINT_SHA256, "Checkpoint bytes differ from the qualified public V1")
    return path


def dependencies():
    import cv2
    import numpy as np
    import pandas as pd
    import pydicom
    try:
        from pydicom.pixels import apply_modality_lut
    except ImportError:
        from pydicom.pixel_data_handlers.util import apply_modality_lut
    import timm
    import torch
    return cv2, np, pd, pydicom, apply_modality_lut, timm, torch


def make_model(checkpoint, device, timm, torch):
    nn = torch.nn

    class FindingAttentionReader(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            self.norm = nn.LayerNorm(backbone.num_features)
            self.att = nn.Sequential(nn.Linear(backbone.num_features, 256), nn.Tanh(),
                                     nn.Dropout(0.2), nn.Linear(256, 12))
            self.clsW = nn.Parameter(torch.zeros(12, backbone.num_features))
            self.clsb = nn.Parameter(torch.zeros(12))
            self.n = 12

        def forward(self, windows):
            batch, count = windows.shape[:2]
            flat = windows.flatten(0, 1)
            encoded = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
            features = self.norm(encoded.view(batch, count, -1))
            attention = torch.softmax(self.att(features), dim=1)
            pooled = torch.einsum("bkn,bkf->bnf", attention, features)
            return (pooled * self.clsW).sum(-1) + self.clsb

    # Tensor-only deserialization is intentionally stricter than the public source.
    # Its compatibility with this checkpoint must pass the native canary first.
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    require(isinstance(state, dict) and "model" in state, "Unsupported checkpoint container")
    require(state.get("arch", ARCH) == ARCH, "Checkpoint architecture differs")
    require(int(state.get("res", CORPUS_RES)) == CORPUS_RES, "Checkpoint training resolution differs")
    backbone = timm.create_model(ARCH, pretrained=False, num_classes=0, in_chans=3,
                                 img_size=INPUT_RES, global_pool="avg")
    model = FindingAttentionReader(backbone)
    model.load_state_dict(state["model"], strict=True)
    del state
    model.eval().to(device)
    require(all(not module.training for module in model.modules()), "Model has a training module")
    return model


def _new_counters():
    return {"missing_slots": 0, "selected_slices": 0, "header_failures": 0, "pixel_failures": 0,
            "study_build_failed": False, "study_build_error_type": None, "study_build_error": None}


def _zero_study(np, counters):
    return (np.zeros((SLICE_COUNT, CORPUS_RES, CORPUS_RES), dtype=np.uint8),
            np.zeros(SLICE_COUNT, dtype=np.uint8), counters)


def build_volume(study_id, rows, series_root, deps):
    """Return (volume, mask, counters). Never raises (reference fault policy).

    Per-header and per-pixel failures are absorbed exactly where the reference
    absorbs them. Any exception that still escapes marks the whole study as
    failed: the volume and mask are zero, as in the reference's study fallback.
    """
    np = deps[1]
    try:
        return _build_volume_checked(study_id, rows, series_root, deps)
    except Exception as error:
        counters = _new_counters()
        counters.update(study_build_failed=True, study_build_error_type=type(error).__name__,
                        study_build_error=str(error)[:300])
        return _zero_study(np, counters)


def _build_volume_checked(study_id, rows, series_root, deps):
    cv2, np, _, pydicom, modality_lut, _, _ = deps
    volume = np.zeros((SLICE_COUNT, CORPUS_RES, CORPUS_RES), dtype=np.uint8)
    used = set()
    offset = 0
    counters = _new_counters()

    def header(path):
        # Reference _hdr: an unreadable header keeps the file as (0.0, path, 0.5, False).
        try:
            data = pydicom.dcmread(path, stop_before_pixels=True)
            orientation = getattr(data, "ImageOrientationPatient", None)
            position = getattr(data, "ImagePositionPatient", None)
            if orientation is not None and position is not None and len(orientation) == 6:
                normal = np.cross(np.asarray(orientation[:3], dtype=float),
                                  np.asarray(orientation[3:], dtype=float))
                coordinate = float(np.dot(np.asarray(position, dtype=float), normal))
            else:
                coordinate = float(getattr(data, "InstanceNumber", 0) or 0)
            spacing = getattr(data, "PixelSpacing", None)
            spacing = float(spacing[0]) if spacing is not None else 0.5
            return coordinate, path, spacing, True
        except Exception:
            return 0.0, path, 0.5, False

    def pixels(item):
        # Reference _px: an unreadable pixel array gives None, and that slice is skipped.
        _, path, spacing, _ = item
        try:
            data = pydicom.dcmread(path)
            image = modality_lut(data.pixel_array, data).astype(np.float32)
            if str(getattr(data, "PhotometricInterpretation", "")) == "MONOCHROME1":
                image = image.max() - image
            return image, spacing
        except Exception:
            return None, spacing

    for plane, fluid, count in SLOTS:
        candidates = [row for row in rows if row["Anatomical_Plane"] == plane
                      and str(row["SeriesInstanceUID"]) not in used]
        preferred = [row for row in candidates
                     if fluid in (0, 1) and int(row.get("Fluid_Sensitive", 0) or 0) == fluid]
        chosen = (preferred or candidates)[0] if candidates else None
        if chosen is None:
            counters["missing_slots"] += 1
            offset += count
            continue
        series_id = str(chosen["SeriesInstanceUID"])
        used.add(series_id)
        # Reference uses glob.glob on the series directory; the same listing order is kept.
        paths = glob.glob(f"{series_root}/{study_id}/{series_id}/*.dcm")
        if not paths:
            counters["missing_slots"] += 1
            offset += count
            continue
        with ThreadPoolExecutor(max_workers=16) as executor:
            records = list(executor.map(header, paths))
        counters["header_failures"] += sum(1 for record in records if not record[3])
        ok_spacings = [record[2] for record in records if record[3]]
        median_spacing = float(np.median(ok_spacings)) if ok_spacings else 0.5
        records.sort(key=lambda record: record[0])
        total = len(records)
        low = int(total * 0.02)
        high = max(int(total * 0.98) - 1, low)
        indices = np.linspace(low, high, count).round().astype(int) if total > 1 else [0] * count
        selected = [records[min(int(index), total - 1)] for index in indices]
        with ThreadPoolExecutor(max_workers=8) as executor:
            arrays = list(executor.map(pixels, selected))
        counters["pixel_failures"] += sum(1 for image, _ in arrays if image is None)
        valid_images = [image for image, _ in arrays if image is not None]
        if valid_images:
            low_value, high_value = np.percentile(np.concatenate([image.ravel() for image in valid_images]),
                                                [2.0, 98.0])
        else:
            low_value, high_value = 0.0, 1.0
        for image, spacing in arrays:
            if image is None:
                offset += 1
                continue
            normalized = np.clip((image - low_value) / (high_value - low_value + 1e-6), 0, 1)
            # A non-2D array fails the crop in the reference (unpacking h, w), so the whole study falls back.
            require(normalized.ndim == 2, "Non-2D DICOM pixel array")
            spacing = spacing if spacing > 0 else median_spacing
            size = min(int(round(CROP_MM / max(spacing, 1e-3))), *normalized.shape)
            require(size > 0, "Empty physical crop")
            top, left = ((normalized.shape[0] - size) // 2, (normalized.shape[1] - size) // 2)
            cropped = normalized[top:top + size, left:left + size]
            resized = cv2.resize(cropped, (CORPUS_RES, CORPUS_RES), interpolation=cv2.INTER_AREA)
            volume[offset] = (resized * 255).astype(np.uint8)
            offset += 1
            counters["selected_slices"] += 1
    require(offset == SLICE_COUNT, "Study slot count differs")
    mask = (volume.reshape(SLICE_COUNT, -1).sum(1) > 0).astype(np.uint8)
    return volume, mask, counters


def prepare_windows(volume, mask, device, mirrored, np, torch):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, len(volume)))
    low, high = int(valid.min()), int(valid.max())
    centers = list(range(low + 1, high))
    if not centers:
        centers = [max(1, min((low + high) // 2, SLICE_COUNT - 2))]
    indices = np.linspace(0, len(centers) - 1, WINDOW_COUNT).round().astype(int)
    centers = np.asarray([centers[int(index)] for index in indices], dtype=np.int64)
    triplets = torch.from_numpy(np.ascontiguousarray(
        volume[np.stack([centers - 1, centers, centers + 1], axis=1)])).to(device)
    require(tuple(triplets.shape) == (WINDOW_COUNT, 3, CORPUS_RES, CORPUS_RES), "Triplet shape differs")
    if mirrored:
        sagittal = torch.from_numpy(np.asarray([
            mirror_axis_for_center(int(center)) == "channels" for center in centers
        ], dtype=bool)).to(device).view(-1, 1, 1, 1)
        # Reflect uint8 acquisition channels first; ImageNet channel statistics differ.
        triplets = torch.where(sagittal, triplets.flip(1), triplets.flip(-1))
    border = (CORPUS_RES - INPUT_RES) // 2
    triplets = triplets[:, :, border:border + INPUT_RES, border:border + INPUT_RES]
    # CPU-generated LUT reproduces the public float32 normalization order exactly.
    values = torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0)
    table = torch.stack([values, values, values], dim=0).view(3, 256, 1)
    means = torch.tensor(IMAGE_MEAN, dtype=torch.float32).view(3, 1, 1)
    scales = torch.tensor(IMAGE_STD, dtype=torch.float32).view(3, 1, 1)
    table = ((table - means) / scales).view(3, 256).contiguous().to(device)
    windows = torch.stack([table[channel][triplets[:, channel].long()] for channel in range(3)], dim=1)
    require(tuple(windows.shape) == (WINDOW_COUNT, 3, INPUT_RES, INPUT_RES), "Model input shape differs")
    return windows


def predict_pair(model, volume, mask, device, np, torch):
    require(all(not module.training for module in model.modules()), "TTA changed model mode")
    predictions = []
    with torch.no_grad():
        for mirrored in (False, True):
            windows = prepare_windows(volume, mask, device, mirrored, np, torch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(windows.unsqueeze(0)).float()
                require(bool(torch.isfinite(logits).all()), "Nonfinite model logits")
                predictions.append(torch.sigmoid(logits)[0].cpu().numpy())
            del windows
    values = 0.5 * (predictions[0] + predictions[1])
    require(values.shape == (12,) and np.isfinite(values).all(), "Invalid study probabilities")
    return values, predictions[0], predictions[1]


def _device_name(torch, index):
    try:
        return torch.cuda.get_device_name(index)
    except Exception:
        return "unavailable"


def run(args):
    started = time.monotonic()
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
    deps = dependencies()
    _, np, pd, _, _, timm, torch = deps
    require(torch.cuda.is_available(), "This experiment requires CUDA/fp16")
    gpu_count = int(torch.cuda.device_count())
    require(gpu_count >= 1, "At least one CUDA device is required")
    device_names = [_device_name(torch, index) for index in range(gpu_count)]
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    checkpoint = find_attached_checkpoint(args.input_root)
    root = args.competition_root
    series_root = root / "test_series"
    if not series_root.is_dir():
        series_root = root / "test_images"
    require(series_root.is_dir(), "Test DICOM directory is absent")
    test = pd.read_csv(root / "test.csv", dtype={"StudyInstanceUID": str})
    sample = pd.read_csv(root / "sample_submission.csv", dtype={"StudyInstanceUID": str})
    all_ids = test["StudyInstanceUID"].tolist()
    expected_columns = ["StudyInstanceUID", *LABELS]
    require(len(set(all_ids)) == len(all_ids), "Duplicate test study ID")
    require(sample.columns.tolist() == expected_columns, "Sample submission label/schema drift")
    require(sample["StudyInstanceUID"].tolist() == all_ids, "Test/sample row order differs")
    ids = all_ids if args.mode == "inference" else all_ids[:args.max_studies]
    require(bool(ids), "No studies selected")
    series = pd.read_csv(root / "test_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
    grouped = {key: frame.to_dict("records") for key, frame in series.groupby("StudyInstanceUID", sort=False)}
    # A study without series rows is predicted from a zero volume, as in the reference (no drop).
    studies_without_series = sum(1 for uid in ids if uid not in grouped)
    models = [make_model(checkpoint, f"cuda:{index}", timm, torch) for index in range(gpu_count)]
    jobs = queue.Queue(maxsize=4)
    results, model_fallbacks, consumer_errors = {}, [], []
    lock = threading.Lock()

    def consumer(index):
        while True:
            item = jobs.get()
            try:
                if item is None:
                    return
                study_id, volume, mask, counters = item
                try:
                    values, plain, mirrored = predict_pair(models[index], volume, mask,
                                                           f"cuda:{index}", np, torch)
                    fallback = None
                except Exception as error:
                    # Reference arm default: a study whose model call fails keeps 0.5 probabilities.
                    values = plain = mirrored = np.full(12, PROBABILITY_DEFAULT, dtype=np.float32)
                    fallback = {"study_id": study_id, "device": f"cuda:{index}",
                                "error_type": type(error).__name__, "message": str(error)[:300]}
                    try:
                        torch.cuda.empty_cache()
                    except Exception:
                        pass
                with lock:
                    results[study_id] = (values, plain, mirrored, counters)
                    if fallback is not None:
                        model_fallbacks.append(fallback)
                    count = len(results)
                    if count % 25 == 0 or count == len(ids):
                        print(json.dumps({"completed": count, "total": len(ids),
                                          "elapsed_s": round(time.monotonic() - started, 2),
                                          "model_fallbacks": len(model_fallbacks)}), flush=True)
            except Exception as error:
                # Keep the consumer alive so the producer is never stranded on a full queue.
                with lock:
                    consumer_errors.append({"study_id": item[0] if item else None,
                                            "error_type": type(error).__name__, "message": str(error)[:300]})
            finally:
                jobs.task_done()

    threads = [threading.Thread(target=consumer, args=(index,)) for index in range(gpu_count)]
    for thread in threads:
        thread.start()
    try:
        with ThreadPoolExecutor(max_workers=args.prep_workers) as executor:
            # At most eight queued preparation futures; no fork after Torch/CUDA initialization.
            for offset in range(0, len(ids), 8):
                batch = ids[offset:offset + 8]
                futures = [executor.submit(build_volume, uid, grouped.get(uid, []), series_root, deps) for uid in batch]
                for uid, future in zip(batch, futures):
                    try:
                        volume, mask, counters = future.result()
                    except Exception as error:
                        volume, mask, counters = _zero_study(np, _new_counters())
                        counters.update(study_build_failed=True, study_build_error_type=type(error).__name__,
                                        study_build_error=str(error)[:300])
                    jobs.put((uid, volume, mask, counters))
    finally:
        for _ in threads:
            jobs.put(None)
        jobs.join()
        for thread in threads:
            thread.join()
    # Coverage guard: a study without a result gets the reference default instead of being dropped.
    missing_results = [uid for uid in ids if uid not in results]
    for uid in missing_results:
        default = np.full(12, PROBABILITY_DEFAULT, dtype=np.float32)
        results[uid] = (default, default, default, _new_counters())
        model_fallbacks.append({"study_id": uid, "device": None, "error_type": "MissingResult",
                                "message": "no consumer result; reference default applied"})
    require(set(results) == set(ids) and len(results) == len(ids), "Incomplete study coverage")
    probabilities = np.stack([results[uid][0] for uid in ids])
    require(probabilities.shape == (len(ids), 12) and np.isfinite(probabilities).all(), "Output shape/finite gate")
    require(bool(((probabilities >= 0) & (probabilities <= 1)).all()), "Probability range gate")
    # Exact public ordinal rank transform; do not quietly substitute a different tie policy.
    ranks = probabilities.argsort(0).argsort(0).astype(np.float64) / max(1, len(ids) - 1)
    # Reference guard: any non-finite rank becomes the neutral 0.5.
    ranks[~np.isfinite(ranks)] = 0.5
    output = pd.DataFrame(ranks.astype(np.float32), columns=LABELS)
    output.insert(0, "StudyInstanceUID", ids)
    require(output.columns.tolist() == expected_columns and output["StudyInstanceUID"].tolist() == ids,
            "Final row/schema gate")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    name = "submission.csv" if args.mode == "inference" else "canary_predictions.csv"
    destination = args.output_dir / name
    require(not destination.exists(), "Output already exists; preserve it and use a fresh directory")
    with destination.open("x") as stream:
        output.to_csv(stream, index=False)
    raw_path = args.output_dir / "probabilities.npz"
    with raw_path.open("xb") as stream:
        np.savez_compressed(stream, study_uids=np.asarray(ids, dtype=str), labels=np.asarray(LABELS),
                            values=probabilities, plain=np.stack([results[uid][1] for uid in ids]),
                            mirrored=np.stack([results[uid][2] for uid in ids]))
    receipt = {"mode": args.mode, "full_test_coverage": ids == all_ids, "studies": len(ids),
               "native_inference_completed": True, "competition_submitted": False,
               "official_score": None, "public_reference_score": 0.949,
               "public_ref": PUBLIC_REF, "public_version": PUBLIC_VERSION,
               "public_numeric_session_id": PUBLIC_SESSION_ID, "checkpoint_sha256": CHECKPOINT_SHA256,
               "dataset_ref": DATASET_REF, "dataset_version": DATASET_VERSION,
               "source_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
               "versions": {"torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__},
               "gpu_count": gpu_count, "devices": device_names,
               "elapsed_s": time.monotonic() - started, "prep_workers": args.prep_workers,
               "fault_policy": FAULT_POLICY,
               "missing_slots": sum(results[uid][3]["missing_slots"] for uid in ids),
               "selected_slices": sum(results[uid][3]["selected_slices"] for uid in ids),
               "header_failures": sum(results[uid][3]["header_failures"] for uid in ids),
               "pixel_failures": sum(results[uid][3]["pixel_failures"] for uid in ids),
               "study_build_failures": sum(1 for uid in ids if results[uid][3]["study_build_failed"]),
               "studies_without_series_metadata": studies_without_series,
               "model_fallback_studies": len(model_fallbacks),
               "model_fallback_examples": model_fallbacks[:5],
               "consumer_error_count": len(consumer_errors), "consumer_error_examples": consumer_errors[:5],
               "prediction_file_sha256": hashlib.sha256(destination.read_bytes()).hexdigest()}
    receipt_path = args.output_dir / "native_inference_receipt.json"
    with receipt_path.open("x") as stream:
        json.dump(receipt, stream, indent=2, sort_keys=True)
        stream.write("\n")
    print(json.dumps(receipt, sort_keys=True), flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--competition-root", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--mode", choices=("canary", "inference"), default="canary")
    parser.add_argument("--max-studies", type=int, default=3)
    parser.add_argument("--prep-workers", type=int, choices=(1, 2, 4), default=2)
    args = parser.parse_args()
    require(1 <= args.max_studies <= 16, "Canary study count must be 1..16")
    run(args)


if __name__ == "__main__":
    main()


## 4. The owned process group

A timeout on one child is not enough. A reader that starts workers can leave descendants behind. A notebook that exports a file before those processes are cleaned up can hide a problem. The next cell writes the owned-process watchdog (sha256 `54636411…`), the same module our private campaign uses. It runs the reader as the leader of a fresh process group, enforces an absolute deadline, and on success terminates the group and verifies that teardown has resolved before the launcher accepts any output. It is Apache-2.0, like the reader.


In [ ]:
%%writefile /kaggle/working/public_v35_source/owned_watchdog.py
# SPDX-License-Identifier: Apache-2.0
"""Bound one owned POSIX process group; never signal a process by its name.

The supervisor holds its group-leader PID until acknowledged. The parent never
reaps it before error/timeout signalling, so that PID cannot be reused while the
ownership lease is held. Descendants that change their process group or session
are outside this contract. By default, success attests only the direct command's
completion. Optional success cleanup terminates its still-owned group before
returning success; it cannot certify absence of escaped descendants. This helper
performs no filesystem cleanup. Deadlines are checked throughout, but POSIX process
creation and other system calls are not themselves interruptibly bounded.
"""
from __future__ import annotations

from dataclasses import dataclass
import json
import math
import os
import select
import signal
import subprocess
import sys
import time


class OwnershipLostError(RuntimeError):
    """Ownership could not be proved; no subsequent group signal is allowed."""


class OwnedProcessError(RuntimeError):
    def __init__(self, message, receipt):
        super().__init__(message)
        self.receipt = receipt


class OwnedProcessTimeout(OwnedProcessError):
    pass


@dataclass(frozen=True)
class DiagnosticBudget:
    remaining_seconds: int
    child_wall_seconds: int
    absolute_monotonic_deadline: float


def diagnostic_budget(native_started_monotonic, *, now=None):
    """Inclusive 850-second parent budget; retain five seconds over the child."""
    current = time.monotonic() if now is None else now
    for value in (native_started_monotonic, current):
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            raise ValueError("diagnostic clock values must be finite numbers")
    elapsed = current - native_started_monotonic
    if elapsed < 0:
        raise ValueError("diagnostic clock moved before its start")
    remaining = math.floor(850 - elapsed)
    if remaining < 65:
        raise ValueError("bootstrap left fewer than 65 seconds for child and watchdog")
    child = remaining - 5
    if not 60 <= child <= 850:
        raise ValueError("child budget violates the native parser contract")
    return DiagnosticBudget(remaining, child, native_started_monotonic + 850)


_SUPERVISOR = r'''
import json, os, signal, subprocess, sys
# A caught TERM handler keeps this ownership anchor alive until group KILL.
# POSIX exec resets caught handlers to default in the actual payload process.
signal.signal(signal.SIGTERM, lambda _number, _frame: None)
status_fd, acknowledgement_fd = int(sys.argv[1]), int(sys.argv[2])
command, payload_fds = json.loads(sys.argv[3]), tuple(json.loads(sys.argv[4]))
def report(value):
    raw = (json.dumps(value, separators=(",", ":")) + "\n").encode()
    if len(raw) > 4096:
        raise RuntimeError("supervisor metadata exceeds pipe bound")
    os.write(status_fd, raw)
try:
    child = subprocess.Popen(command, pass_fds=payload_fds)
    report({"stage": "started", "command_pid": child.pid})
    result = child.wait()
    report({"stage": "finished", "returncode": result})
except BaseException as exc:
    report({"stage": "supervisor_error", "error_type": type(exc).__name__})
# Hold the unreaped group-leader identity through the parent's decision.
acknowledgement = os.read(acknowledgement_fd, 1)
sys.exit(0 if acknowledgement == b"S" else 125)
'''


class _Lease:
    def __init__(self, process):
        self.process = process
        self.pid = process.pid
        self.signals = []
        self.verify()

    def verify(self):
        if self.process.returncode is not None:
            raise OwnershipLostError("owned leader was already reaped")
        try:
            sid, pgid = os.getsid(self.pid), os.getpgid(self.pid)
        except OSError as exc:
            raise OwnershipLostError("owned leader identity unavailable") from exc
        if sid != self.pid or pgid != self.pid:
            raise OwnershipLostError("owned leader session/group identity changed")
        return {"session_id": sid, "process_group_id": pgid}

    def send_group(self, number):
        proof = self.verify()
        event = {"signal": signal.Signals(number).name,
                 "target_process_group_id": self.pid,
                 "observed_session_id": proof["session_id"],
                 "observed_process_group_id": proof["process_group_id"]}
        try:
            os.killpg(self.pid, number)
        except ProcessLookupError:
            self.signals.append({**event, "group_present": False})
        else:
            self.signals.append({**event, "group_present": True})


def _terminate_owned(lease, deadline):
    """Keep the leader unreaped until both bounded group signals are attempted."""
    record = {"ownership_proved": False, "signals": [], "leader_reaped": False,
              "unresolved": False}
    try:
        lease.verify()
        record["ownership_proved"] = True
        lease.send_group(signal.SIGTERM)
        grace = min(0.25, max(0.0, deadline - time.monotonic()) / 4)
        if grace:
            time.sleep(grace)
        # The still-unreaped leader reserves the group ID even after SIGTERM.
        lease.send_group(signal.SIGKILL)
        record["signals"] = list(lease.signals)
        remaining = max(0.0, deadline - time.monotonic())
        lease.process.wait(timeout=remaining)
        record["leader_reaped"] = True
    except (OwnershipLostError, subprocess.TimeoutExpired, OSError, ChildProcessError) as exc:
        record["signals"] = list(lease.signals)
        record["unresolved"] = True
        record["error_type"] = type(exc).__name__
        # Ownership loss is a hard stop, never a reason for a fallback signal.
    return record


def run_owned(command, *, timeout, absolute_deadline=None, cleanup_on_success=False,
              cwd=None, env=None, stdout=None, stderr=None, passthrough_fds=()):
    """Run a command in a fresh owned session, preserving an inclusive deadline.

Return a bounded metadata receipt on success. Nonzero exit, timeout or an
observer error stops only the freshly owned group and raises with its receipt.
With cleanup_on_success=True, keep the successful command's supervisor
unacknowledged and unreaped through TERM and KILL, and require resolved teardown
before declaring direct-command success. The default retains the original
acknowledgement-only behavior. No group signal is issued after acknowledgement
or reaping, and an unresolved teardown is never retried.
SIGCHLD must retain its default disposition; callers must not reap this private
supervisor from another thread or install a reaping handler during this call.
"""
    if os.name != "posix" or not hasattr(os, "killpg"):
        raise RuntimeError("owned process-group watchdog requires POSIX")
    if signal.getsignal(signal.SIGCHLD) != signal.SIG_DFL:
        raise OwnershipLostError("default SIGCHLD disposition is required")
    if type(cleanup_on_success) is not bool:
        raise ValueError("cleanup_on_success must be a boolean")
    if isinstance(timeout, bool) or not isinstance(timeout, (int, float)) or not math.isfinite(timeout) or timeout <= 0.25:
        raise ValueError("watchdog timeout must be finite and greater than 0.25 seconds")
    if not isinstance(command, (list, tuple)) or not command or any(not isinstance(part, str) or not part for part in command):
        raise ValueError("command must be a nonempty argument vector")
    fds = tuple(passthrough_fds)
    if any(isinstance(fd, bool) or not isinstance(fd, int) or fd < 3 for fd in fds) or len(set(fds)) != len(fds):
        raise ValueError("passthrough file descriptors must be unique integers at least three")
    started = time.monotonic()
    deadline = started + timeout
    if absolute_deadline is not None:
        if isinstance(absolute_deadline, bool) or not isinstance(absolute_deadline, (int, float)) or not math.isfinite(absolute_deadline):
            raise ValueError("absolute watchdog deadline must be a finite monotonic clock value")
        deadline = min(deadline, absolute_deadline)
    effective_timeout = deadline - started
    if effective_timeout <= 0.25:
        raise ValueError("absolute deadline left insufficient watchdog time before spawn")
    reserve = min(2.0, max(0.2, effective_timeout / 5))
    observation_deadline = deadline - reserve
    status_read, status_write = os.pipe()
    acknowledgement_read, acknowledgement_write = os.pipe()
    process = lease = None
    receipt = {"status": "UNRESOLVED", "ownership_strategy": "unreaped-owned-supervisor",
               "command_returncode": None, "timed_out": False, "group_signals": [],
               "effective_timeout_seconds": effective_timeout,
               "absolute_monotonic_deadline": deadline,
               "cleanup_on_success": cleanup_on_success}
    supervisor_acknowledged = False
    teardown_attempted = False
    closed = set()
    def close_fd(fd):
        if fd not in closed:
            os.close(fd)
            closed.add(fd)
    try:
        process = subprocess.Popen(
            [sys.executable, "-u", "-c", _SUPERVISOR, str(status_write),
             str(acknowledgement_read), json.dumps(list(command)), json.dumps(fds)],
            cwd=cwd, env=env, stdout=stdout, stderr=stderr,
            start_new_session=True, pass_fds=(status_write, acknowledgement_read, *fds),
        )
        close_fd(status_write)
        close_fd(acknowledgement_read)
        lease = _Lease(process)
        receipt.update({"owned_leader_pid": lease.pid, "owned_session_id": lease.pid,
                        "owned_process_group_id": lease.pid})
        os.set_blocking(status_read, False)
        buffered = b""
        command_started = False
        while True:
            left = observation_deadline - time.monotonic()
            if left <= 0:
                receipt["timed_out"] = True
                raise TimeoutError("owned command exceeded its observation deadline")
            ready, _, _ = select.select([status_read], [], [], min(left, 0.05))
            if not ready:
                continue
            raw = os.read(status_read, 4096)
            if not raw:
                raise RuntimeError("owned supervisor metadata pipe closed before completion")
            buffered += raw
            if len(buffered) > 8192:
                raise RuntimeError("owned supervisor metadata exceeded its bound")
            while b"\n" in buffered:
                line, buffered = buffered.split(b"\n", 1)
                message = json.loads(line)
                if message.get("stage") == "started":
                    if command_started or set(message) != {"stage", "command_pid"} or type(message["command_pid"]) is not int:
                        raise RuntimeError("invalid owned-command start metadata")
                    command_started = True
                    receipt["command_pid"] = message["command_pid"]
                elif message.get("stage") == "finished":
                    if not command_started or set(message) != {"stage", "returncode"} or type(message["returncode"]) is not int:
                        raise RuntimeError("invalid owned-command completion metadata")
                    receipt["command_returncode"] = message["returncode"]
                    if message["returncode"] != 0:
                        raise RuntimeError("owned command returned a nonzero status")
                    lease.verify()
                    if cleanup_on_success:
                        # The zero-exit payload is reaped by the supervisor, while
                        # this fresh session leader still anchors group ownership.
                        teardown_attempted = True
                        teardown = _terminate_owned(lease, deadline)
                        receipt["teardown"] = teardown
                        receipt["group_signals"] = teardown["signals"]
                        if (teardown["unresolved"] or not teardown["ownership_proved"]
                                or not teardown["leader_reaped"]):
                            if teardown.get("error_type") == "TimeoutExpired":
                                raise TimeoutError("owned success cleanup exceeded its deadline")
                            raise RuntimeError("owned success cleanup did not resolve group teardown")
                        completed = time.monotonic()
                        if completed > deadline:
                            raise TimeoutError("owned success cleanup completed after its inclusive deadline")
                        receipt.update({"status": "PASS_OWNED_COMMAND", "leader_reaped": True,
                                        "success_completion_strategy": "direct-zero-exit-owned-group-teardown",
                                        "supervisor_acknowledged": False,
                                        "supervisor_returncode": process.returncode,
                                        "elapsed_seconds": completed - started})
                        return receipt
                    os.write(acknowledgement_write, b"S")
                    supervisor_acknowledged = True
                    process.wait(timeout=max(0.0, deadline - time.monotonic()))
                    if process.returncode != 0:
                        raise RuntimeError("owned supervisor did not acknowledge success")
                    completed = time.monotonic()
                    if completed > deadline:
                        raise TimeoutError("owned command acknowledgement completed after its inclusive deadline")
                    receipt.update({"status": "PASS_OWNED_COMMAND", "leader_reaped": True,
                                    "elapsed_seconds": completed - started})
                    return receipt
                else:
                    raise RuntimeError("owned supervisor reported an error or unknown stage")
    except BaseException as exc:
        if isinstance(exc, (TimeoutError, subprocess.TimeoutExpired)):
            receipt["timed_out"] = True
        # An acknowledged supervisor may already be reaped; never signal it.
        if lease is not None and not supervisor_acknowledged and not teardown_attempted:
            teardown_attempted = True
            teardown = _terminate_owned(lease, deadline)
            receipt["teardown"] = teardown
            receipt["group_signals"] = teardown["signals"]
        elif process is not None and "teardown" not in receipt:
            receipt["teardown"] = {"unresolved": True, "reason": "ownership unavailable or success already acknowledged"}
        receipt.update({"status": "TIMEOUT_OWNED_COMMAND" if receipt["timed_out"] else "FAIL_OWNED_COMMAND",
                        "error_type": type(exc).__name__, "elapsed_seconds": time.monotonic() - started})
        failure = OwnedProcessTimeout if receipt["timed_out"] else OwnedProcessError
        raise failure("owned watchdog stopped without declaring native success", receipt) from exc
    finally:
        for fd in (status_read, status_write, acknowledgement_read, acknowledgement_write):
            close_fd(fd)


## 5. The second arm: goodpjw2008's ConvNeXt 2.5D reader

goodpjw2008 published a three-fold ConvNeXt reader (dataset [goodpjw2008 / rsna-knee-2-5d-convnext-reader](https://www.kaggle.com/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader), version 1, Apache-2.0). Its released sources are `preprocess.py`, `knee.py` and `infer.py`. The next four cells write them to disk byte for byte, and the launcher checks each file against its published hash. The reader turns each DICOM series into a canonical volume, takes twelve three-slice windows per slot over six slots, runs the three checkpoints of `convnext_tiny.fb_in22k_ft_in1k`, and averages their sigmoid outputs. The ConvNeXt glue cell adds an output directory, a per-study fault flag and a decode report. It never reads labels and never trains.

**Why a 0.30 weight.** The predeclared weight is 0.30. It is one global weight for all twelve findings, chosen before any run, and it is not tuned on the public board or on the gold studies. A public notebook that blends this reader at about the same weight reports a similar small change on its own base.

**Failures.** A study whose input or model call fails, or whose selected series has a slice or series that cannot be decoded, is left at 0.5 for this arm and counted; it is never fatal. If more than half the studies fail, or the child process fails or times out, the arm is marked failed and the notebook exports the mirror arm alone. The decoder step installs each pylibjpeg wheel on its own. The pure-Python frontend alone decodes nothing; a binary plugin is needed for compressed transfer syntaxes. The receipt records each wheel's outcome, the Python version and which transfer syntaxes this runtime can decode.

**Credits and licences.** The released reader, its sources and the dataset are goodpjw2008's, Apache-2.0. The backbone is `convnext_tiny.fb_in22k_ft_in1k` (ImageNet-22k, then ImageNet-1k). The decoder packages keep their own terms: pylibjpeg MIT; pylibjpeg-openjpeg MIT with bundled notices; pylibjpeg-libjpeg GPLv3 with a proprietary alternative. The blend, the receipts and this notebook's orchestration are original and released under MIT, copyright 2026 cubres.


In [ ]:
%%writefile /kaggle/working/public_v35_source/convnext_child.py
# SPDX-License-Identifier: Apache-2.0
"""Original glue for the goodpjw2008 ConvNeXt 2.5D arm (candidate_v17).

The arm calls the three released source modules (preprocess.py, knee.py, infer.py), which the parent has
byte-pinned. Their DICOM preprocessing, windows, three checkpoints and sigmoid fold average are used as published.
This glue adds: a fresh output directory; a per-study input flag; a per-study model fallback; a decoder capability
probe (which pydicom transfer syntaxes this runtime can decode, and the Python version); and per-study decode
accounting. A study with any decode failure in its selected series (a slice or a series) is left at 0.5 for this arm
and counted (rule DECODE_FAILURE_RULE). The optional visible comparison with the author's probabilities is kept.
It reads no labels and trains nothing. Released sources: goodpjw2008, Apache-2.0.
"""
import argparse
import contextlib
import hashlib
import importlib.util
import json
import multiprocessing
import os
import sys
import time
from pathlib import Path

K_WINDOWS = 12
BATCH_SIZE = 4
# A slice or series that fails to decode makes its study 0.5 for this arm (counted).
# "any_decode_failure_is_study_half": any failed slice or series in a selected series (the default).
# "series_failure_is_study_half": only a series with no readable slice counts; failed slices are counted and skipped.
DECODE_FAILURE_RULE = "any_decode_failure_is_study_half"
DECODER_CLASSES = ["RLELosslessDecoder", "JPEGBaseline8BitDecoder", "JPEGExtended12BitDecoder", "JPEGLosslessDecoder",
                   "JPEGLosslessSV1Decoder", "JPEG2000Decoder", "JPEG2000LosslessDecoder", "JPEGLSLosslessDecoder",
                   "JPEGLSNearLosslessDecoder", "HTJ2KDecoder", "HTJ2KLosslessDecoder"]
PLUGIN_MODULES = ["pylibjpeg", "libjpeg", "openjpeg", "gdcm", "pyjpegls", "PIL"]


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def require(value, message):
    if not value:
        raise RuntimeError(message)


def save_new(path, value):
    with Path(path).open("x") as stream:
        json.dump(value, stream, indent=2, sort_keys=True, allow_nan=False)
        stream.write("\n")


def decoder_probe(pydicom):
    """Which pydicom transfer syntaxes this runtime can decode, with the Python and plugin versions."""
    from pydicom.pixels import decoders
    rows = {}
    for name in DECODER_CLASSES:
        decoder = getattr(decoders, name, None)
        if decoder is None:
            rows[name] = dict(present=False)
            continue
        available = bool(decoder.is_available)
        rows[name] = dict(present=True, uid=str(decoder.UID), available=available,
                          missing=[] if available else [str(m) for m in decoder.missing_dependencies])
    return dict(sys_version=sys.version, pydicom=pydicom.__version__,
                decoders=rows, plugin_modules={m: bool(importlib.util.find_spec(m)) for m in PLUGIN_MODULES},
                available_transfer_syntaxes=sorted(k for k, v in rows.items() if v.get("available")))


def main():
    began = time.monotonic()
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    config = json.loads(Path(args.config).read_text())
    output = Path(config["output_dir"])
    output.mkdir(parents=True, exist_ok=False)
    receipt = dict(status="STARTED", arm="goodpjw2008 ConvNeXt 2.5D reader V1 (author notebook V2)",
                   labels_read=False, training=False, decode_failure_rule=DECODE_FAILURE_RULE)
    try:
        for item in config["file_pins"]:
            path = Path(item["path"])
            require(path.is_file() and path.stat().st_size == item["bytes"], "Pinned size differs: " + path.name)
            require(sha256_file(path) == item["sha256"], "Pinned hash differs: " + path.name)
        import numpy as np
        import pandas as pd
        import pydicom
        import torch
        import infer
        import knee
        import preprocess
        source_dir = Path(config["source_dir"]).resolve()
        for module in (infer, knee, preprocess):
            require(Path(module.__file__).resolve().parent == source_dir, "Released module origin differs")
        labels = list(knee.LABELS)
        require(labels == list(config["labels"]), "Finding order differs")
        probe = decoder_probe(pydicom)
        receipt["probe"] = probe
        device = config["device"]
        on_cuda = device.startswith("cuda")
        if on_cuda:
            require(torch.cuda.is_available(), "CUDA device required")
        ids = list(config["study_ids"])
        require(ids and len(ids) == len(set(ids)), "Study identities must be nonempty and unique")
        root = Path(config["competition_dir"])
        series = pd.read_csv(root / "test_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
        series = series[series["StudyInstanceUID"].isin(set(ids))]
        series_root = root / "test_series"
        n_files = {}
        for study, sid in zip(series["StudyInstanceUID"], series["SeriesInstanceUID"]):
            directory = series_root / study / sid
            n_files[sid] = min(len(os.listdir(directory)), preprocess.MAX_SLICES) if directory.is_dir() else 0
        table = knee.build_study_table(series, n_files)

        # Decode accounting, per study index, in shared memory so DataLoader workers can write to it.
        position = {sid: i for i, sid in enumerate(ids)}
        series_failed = multiprocessing.Array("i", len(ids), lock=True)
        slices_failed = multiprocessing.Array("i", len(ids), lock=True)
        original_load_series = infer.load_series

        def counting_load_series(sdir, *args, **kwargs):
            index = position.get(Path(sdir).parent.name)
            try:
                volume, info = original_load_series(sdir, *args, **kwargs)
            except Exception:
                if index is not None:
                    with series_failed.get_lock():
                        series_failed[index] += 1
                raise
            if index is not None:
                if int(info.get("n_raw", 0)) == 0:
                    with series_failed.get_lock():
                        series_failed[index] += 1
                bad = int(info.get("n_bad", 0))
                if bad:
                    with slices_failed.get_lock():
                        slices_failed[index] += bad
            return volume, info

        infer.load_series = counting_load_series
        models = infer.load_models(config["checkpoints"], device=device)

        class GuardedStudies(infer.DicomStudyDataset):
            """Released DicomStudyDataset. An exception for one study is flagged, never raised into the batch.
            Codes: 1 usable, 0 input exception, 2 decode failure in a selected series (see DECODE_FAILURE_RULE)."""

            def __getitem__(self, index):
                try:
                    x, mask, pos = super().__getitem__(index)
                except Exception:
                    return (torch.zeros((knee.N_SLOTS, K_WINDOWS, 3, preprocess.SIZE, preprocess.SIZE), dtype=torch.uint8),
                            torch.zeros(knee.N_SLOTS, dtype=torch.bool),
                            torch.zeros((knee.N_SLOTS, K_WINDOWS), dtype=torch.float32),
                            torch.tensor(0, dtype=torch.int8))
                failed = series_failed[index] > 0
                if DECODE_FAILURE_RULE == "any_decode_failure_is_study_half":
                    failed = failed or slices_failed[index] > 0
                return x, mask, pos, torch.tensor(2 if failed else 1, dtype=torch.int8)

        loader = torch.utils.data.DataLoader(GuardedStudies(ids, table, str(series_root), K_WINDOWS),
                                             batch_size=BATCH_SIZE, num_workers=int(config["workers"]),
                                             pin_memory=on_cuda)
        autocast = torch.autocast("cuda", dtype=torch.float16) if on_cuda else contextlib.nullcontext()

        def forward(x, mask, pos):
            x, mask, pos = x.to(device, non_blocking=True), mask.to(device), pos.to(device)
            with torch.no_grad(), autocast:
                p = torch.stack([m(x, mask, pos, res=res).float().sigmoid() for m, res in models]).mean(0)
            values = p.float().cpu().numpy().astype(np.float64)
            require(np.isfinite(values).all(), "Nonfinite reader output")
            return values

        values = np.full((len(ids), len(labels)), 0.5, np.float64)
        # 0 ok, 1 study input exception, 2 model failed, 3 decode failure (all three left at 0.5 for this arm)
        status = np.zeros(len(ids), np.int8)
        cursor, batch_seconds = 0, []
        for x, mask, pos, code in loader:
            count = x.shape[0]
            rows = list(range(cursor, cursor + count))
            cursor += count
            started = time.monotonic()
            good = []
            for j in range(count):
                c = int(code[j])
                if c == 1:
                    good.append(j)
                elif c == 0:
                    status[rows[j]] = 1
                else:
                    status[rows[j]] = 3
            if good:
                index = torch.tensor(good)
                try:
                    values[[rows[j] for j in good]] = forward(x[index], mask[index], pos[index])
                except Exception:
                    for j in good:
                        try:
                            values[rows[j]] = forward(x[j:j + 1], mask[j:j + 1], pos[j:j + 1])[0]
                        except Exception:
                            status[rows[j]] = 2
            batch_seconds.append(time.monotonic() - started)
        require(cursor == len(ids), "Loader cursor differs from the study count")
        slices_array = np.asarray(list(slices_failed), dtype=np.int64)
        series_array = np.asarray(list(series_failed), dtype=np.int64)
        np.savez_compressed(output / "convnext_probabilities.npz", study_uids=np.asarray(ids, dtype=str),
                            labels=np.asarray(labels), values=values, status=status,
                            decode_failed_slices=slices_array, decode_failed_series=series_array)
        frame = pd.DataFrame(values.astype(np.float32), columns=labels)
        frame.insert(0, "StudyInstanceUID", ids)
        with (output / "convnext_probabilities.csv").open("x") as stream:
            frame.to_csv(stream, index=False)
        oracle_report = None
        if config.get("oracle"):
            oracle = config["oracle"]
            position_map = {sid: i for i, sid in enumerate(ids)}
            rows = [position_map[sid] for sid in oracle["study_ids"] if sid in position_map]
            reference = np.asarray([oracle["probabilities"][oracle["study_ids"].index(ids[r])] for r in rows], np.float64)
            difference = float(np.abs(values[rows] - reference).max()) if rows else float("inf")
            oracle_report = dict(studies=len(rows), max_abs_diff=difference, tolerance=float(oracle["tolerance"]),
                                 all_studies_ok=bool((status[rows] == 0).all()) if rows else False,
                                 passed=bool(rows and difference <= float(oracle["tolerance"]) and (status[rows] == 0).all()))
        decode_block = dict(rule=DECODE_FAILURE_RULE, decode_failed_studies=int((status == 3).sum()),
                            studies_with_series_failure=int((series_array > 0).sum()),
                            studies_with_slice_failure=int((slices_array > 0).sum()),
                            series_failed_total=int(series_array.sum()), slices_failed_total=int(slices_array.sum()),
                            available_transfer_syntaxes=probe["available_transfer_syntaxes"])
        receipt.update(status="PASS_CONVNEXT_ARM", studies=len(ids), ok_studies=int((status == 0).sum()),
                       input_failed=int((status == 1).sum()), model_failed=int((status == 2).sum()),
                       decode=decode_block, oracle=oracle_report,
                       batch_seconds_mean=float(np.mean(batch_seconds)) if batch_seconds else 0.0,
                       batch_seconds_max=float(np.max(batch_seconds)) if batch_seconds else 0.0,
                       elapsed_seconds=time.monotonic() - began, torch=torch.__version__, numpy=np.__version__,
                       pandas=pd.__version__, pydicom=pydicom.__version__, python=sys.version, device=device,
                       device_name=(torch.cuda.get_device_name(0) if on_cuda else None),
                       checkpoint_sha256=[sha256_file(p) for p in config["checkpoints"]])
        save_new(output / "receipt.json", receipt)
        print(json.dumps({k: receipt[k] for k in ("status", "studies", "ok_studies", "input_failed", "model_failed",
                                                  "elapsed_seconds")} | {"decode_failed_studies": decode_block["decode_failed_studies"]}),
              flush=True)
    except BaseException as error:
        receipt.update(status="FAIL_CONVNEXT_ARM", error_type=type(error).__name__, error=str(error)[:500],
                       elapsed_seconds=time.monotonic() - began)
        if not (output / "receipt.json").exists():
            save_new(output / "receipt.json", receipt)
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/public_v35_source/preprocess.py
"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].

Canonical means: fixed physical pixel size, centre crop to a fixed field of view,
standard in-plane orientation, standard slice order and a per-series intensity window.
The same `load_series` is used for the training cache and for inference.

Usage: python src/preprocess.py [train|test] [n_series_limit]
"""
import os
import sys
from multiprocessing import Pool

import cv2
import numpy as np
import pandas as pd
import pydicom

DATA = "/mnt/ssd/rsna_knee/data"
WORK = "/mnt/ssd/rsna_knee/work"

MM_PER_PX = 0.4
SIZE = 384  # 153.6 mm field of view
MAX_SLICES = 64

# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates
# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior
# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior
# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior
CANON = {
    0: (+1, (1, +1), (2, -1)),
    1: (+1, (0, +1), (2, -1)),
    2: (-1, (0, +1), (1, +1)),
}
PLANES = ["Sagittal", "Coronal", "Axial"]


def _read(path):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
    if slope is not None and inter is not None:
        arr = arr * float(slope) + float(inter)
    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None
    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None
    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None
    inst = float(ds.get("InstanceNumber") or 0)
    frames = list(arr) if arr.ndim == 3 else [arr]
    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]


def _canon_slice(img, iop, ps, plane):
    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""
    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[plane]
    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]
    if c[rax] * rsgn < 0:
        img = img[::-1]
    return img, ps


def _resample(img, ps):
    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""
    out = np.zeros((SIZE, SIZE), np.float32)
    h, w = img.shape
    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))
    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))
    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def load_series(sdir, plane_name=None):
    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""
    slices, bad = [], 0
    for fn in os.listdir(sdir):
        try:
            slices.extend(_read(os.path.join(sdir, fn)))
        except Exception:
            bad += 1
    info = {"n_raw": len(slices), "n_bad": bad}
    if not slices:
        return np.zeros((1, SIZE, SIZE), np.uint8), info

    iop = next((s[1] for s in slices if s[1] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        plane = int(np.abs(normal).argmax())
    else:
        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0
    info["plane"] = PLANES[plane]

    if normal is not None and all(s[2] is not None for s in slices):
        sgn = CANON[plane][0] * np.sign(normal[plane])
        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)
        pos = [float(np.dot(s[2], normal)) for s in slices]
        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0
    else:
        slices.sort(key=lambda s: s[4])
    if len(slices) > MAX_SLICES:
        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)
        slices = [slices[i] for i in keep]

    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])
    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    info["n"] = len(vol)
    return (vol * 255).round().astype(np.uint8), info


def _job(args):
    split, study, series, plane, out_dir = args
    out = f"{out_dir}/{series}.npy"
    info = {"study": study, "series": series}
    if os.path.exists(out):
        return None
    try:
        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)
        np.save(out + ".tmp.npy", vol)
        os.replace(out + ".tmp.npy", out)
        info.update(extra)
    except Exception as e:
        info["error"] = f"{type(e).__name__}: {e}"[:200]
    return info


def main():
    split = sys.argv[1] if len(sys.argv) > 1 else "train"
    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None
    out_dir = f"{WORK}/cache{SIZE}_{split}"
    os.makedirs(out_dir, exist_ok=True)
    se = pd.read_csv(f"{DATA}/{split}_series.csv")
    if limit:
        se = se.sample(limit, random_state=0)
    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]
    rows = []
    with Pool(18) as p:
        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):
            if r:
                rows.append(r)
            if i % 1000 == 0:
                print(f"{i}/{len(tasks)}", flush=True)
    df = pd.DataFrame(rows)
    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"
    if os.path.exists(meta) and len(df):
        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")
    if len(df):
        df.to_csv(meta, index=False)
        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/public_v35_source/knee.py
"""Shared dataset and model code for training and inference."""
import math

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first
N_SLOTS = 6
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def slot_of(plane, fat_sup):
    return PLANES.index(plane) + 3 * (1 - int(fat_sup))


def build_study_table(series_df, n_slices):
    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.
    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}
    with each slot's candidates sorted by slice count (largest first)."""
    table = {}
    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,
                              series_df.Fat_Suppression, series_df.Anatomical_Plane):
        if n_slices.get(sr, 0) < 3:
            continue
        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)
    for slots in table.values():
        for cands in slots:
            cands.sort(key=lambda s: -n_slices[s])
    return table


def window_centres(n, k, train, lo=0.04, hi=0.96):
    """k slice indices (centres of 3-slice windows) spread over the series."""
    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    if train:
        edges = np.linspace(a, b, k + 1)
        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])
    else:
        c = np.linspace(a, b, k)
    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))


class StudyDataset(torch.utils.data.Dataset):
    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""

    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):
        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir
        self.targets, self.k, self.train = targets, k, train
        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]
            vol = self.loader(sr)
            n = len(vol)
            c = window_centres(n, self.k, self.train)
            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]
            if x is None:
                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)
            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)
        if x is None:
            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)
        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))


def make_views(x, res, train, group, fov=0.92):
    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached
    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter
    that is shared within a series."""
    n = x.shape[0]
    ns = n // group
    dev = x.device
    x = x.float().div_(255.0)
    if train:
        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)
        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)
        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12
        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12
        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)
        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2
    else:
        ang = torch.zeros(ns, device=dev)
        sc = torch.full((ns,), fov, device=dev)
        tx = ty = torch.zeros(ns, device=dev)
        gamma = gain = None
    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    theta = theta.repeat_interleave(group, 0)
    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)
    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot
        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    if train:
        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)
        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std


class KneeNet(nn.Module):
    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->
    small transformer over all windows of the study -> one attention pooling per finding."""

    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1):
        super().__init__()
        kw = {"drop_path_rate": drop_path} if drop_path else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(N_SLOTS, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))
        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))

    def head(self, feat, mask, pos):
        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""
        b, s, k, _ = feat.shape
        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)
        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))
        h = h.view(b, s * k, -1)
        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)
        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

    def forward(self, x, mask, pos, res=256, train=False):
        """x uint8 [B, S, K, 3, H, W]."""
        b, s, k = x.shape[:3]
        keep = mask.view(-1)  # run the backbone only on series that exist
        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)
        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))
        feat = f.new_zeros(b * s, k, f.shape[-1])
        feat[keep] = f.view(-1, k, f.shape[-1])
        return self.head(feat.view(b, s, k, -1), mask, pos)

In [ ]:
%%writefile /kaggle/working/public_v35_source/infer.py
"""Predict the 12 findings for every study straight from DICOM folders.

Used by the Kaggle submission notebook and locally to check that predictions from raw DICOMs
match those from the training cache.

Usage: python src/infer.py <data_dir> <split> <out_csv> <ckpt> [<ckpt> ...]
  data_dir contains <split>.csv, <split>_series.csv and <split>_series/
"""
import os
import sys

import numpy as np
import pandas as pd
import torch

from knee import LABELS, N_SLOTS, KneeNet, build_study_table, window_centres
from preprocess import MAX_SLICES, SIZE, load_series


class DicomStudyDataset(torch.utils.data.Dataset):
    """Loads every selected series of a study once; windows are cut per model setting later."""

    def __init__(self, studies, table, root, k):
        self.studies, self.table, self.root, self.k = list(studies), table, root, k

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x = np.zeros((N_SLOTS, self.k, 3, SIZE, SIZE), np.uint8)
        mask, pos = np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            try:
                vol, _ = load_series(f"{self.root}/{st}/{cands[0]}")
            except Exception:
                continue
            n = len(vol)
            if n < 3:
                continue
            c = window_centres(n, self.k, False)
            x[s] = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])
            mask[s], pos[s] = True, c / max(n - 1, 1)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos)


def load_models(ckpts, device="cuda"):
    models = []
    for path in ckpts:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        # the pretrained tag after the dot is irrelevant here and unknown to older timm versions
        m = KneeNet(ck["args"]["backbone"].split(".")[0], pretrained=False)
        m.load_state_dict(ck["model"])
        models.append((m.to(device).to(memory_format=torch.channels_last).eval(), ck["args"]["res"]))
    return models


@torch.no_grad()
def run(data_dir, split, ckpts, k=12, studies=None, workers=4, bs=4):
    se = pd.read_csv(f"{data_dir}/{split}_series.csv")
    root = f"{data_dir}/{split}_series"
    if studies is None:
        studies = pd.read_csv(f"{data_dir}/{split}.csv").StudyInstanceUID.tolist()
    se = se[se.StudyInstanceUID.isin(set(studies))]
    n_files = {}
    for st, sr in zip(se.StudyInstanceUID, se.SeriesInstanceUID):
        d = f"{root}/{st}/{sr}"
        n_files[sr] = min(len(os.listdir(d)), MAX_SLICES) if os.path.isdir(d) else 0
    table = build_study_table(se, n_files)
    dl = torch.utils.data.DataLoader(DicomStudyDataset(studies, table, root, k), batch_size=bs,
                                     num_workers=workers, pin_memory=True)
    models = load_models(ckpts)
    out = []
    for x, mask, pos in dl:
        x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            p = torch.stack([m(x, mask, pos, res=res).float().sigmoid() for m, res in models]).mean(0)
        out.append(p.cpu())
    return pd.DataFrame(torch.cat(out).numpy(), index=pd.Index(studies, name="StudyInstanceUID"), columns=LABELS)


if __name__ == "__main__":
    data_dir, split, out_csv, ckpts = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4:]
    run(data_dir, split, ckpts).to_csv(out_csv)
    print("wrote", out_csv)

## 4. Let each finding choose its windows

CoAtNet encodes the triplets in chunks of 16. LayerNorm and a small attention head give each finding its own softmax over the windows. The weighted sum of window features produces one vector per finding; a finding-specific classifier turns it into a logit. Dropout is disabled in evaluation mode.

```text
attention[k, c] = softmax over k (attention_head(feature[k]))
pooled[c]       = sum over k (attention[k, c] * feature[k])
logit[c]        = classifier[c](pooled[c])
probability[c]  = (sigmoid(plain_logit[c]) + sigmoid(mirror_logit[c])) / 2
```

The final CSV uses the upstream per-column `argsort().argsort() / max(1, N-1)` transform. These ordinal ranks are useful for ranking; they are not calibrated probabilities. Preserve the source's NumPy tie behavior. Canary ranks depend on the canary's N and therefore cannot establish full-test prediction equivalence.


## 6. Run in stages and export last

The launcher checks the pinned runtime, the reader, the watchdog, the ConvNeXt glue and each released module against its hash before any model is loaded. It runs the mirror arm first, under the owned watchdog with the absolute deadline set in the first cell. The mirror output is validated: receipt, coverage, schema, row order, finiteness, the checkpoint hash, and the recomputed ranks. Then the ConvNeXt arm runs if the time gate allows. Its child process has its own receipt, and the blend is computed from both arms' probabilities. The staged blend, the mirror-only file and the arm receipts go to a fresh stage directory, not to the submission path. Only after every check does the notebook write `/kaggle/working/submission.csv`, with exclusive creation.

The checkpoints require the publisher's weights dataset attachment, and the ConvNeXt arm requires the goodpjw dataset attachment. The reader hashes the actual checkpoint file before deserialization. No code license grants rights to redistribute checkpoint bytes or controlled-access records.


In [ ]:
import importlib.metadata
import importlib.util
import io
import subprocess

import numpy as np
import pandas as pd

# Pins first. Distribution metadata only: no CUDA or model work happens in this cell.
require(importlib.metadata.version("torch") == CONFIG["EXPECTED_TORCH"] and
        importlib.metadata.version("timm") == CONFIG["EXPECTED_TIMM"],
        "Use the qualified Torch 2.11.0+cu128 / timm 1.0.29 container")


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def sha256_bytes(raw):
    return hashlib.sha256(raw).hexdigest()


require(sha256_file(READER_PATH) == READER_SHA256, "Staged reader bytes differ from the qualified reader")
require(sha256_file(WATCHDOG_PATH) == OWNED_WATCHDOG_SHA256, "Staged owned watchdog differs from its pin")
require(sha256_file(CHILD_PATH) == CHILD_SHA256, "Staged ConvNeXt glue differs from its pin")
# IPython %%writefile ends every file with a newline; the published released modules end without one.
# Restore the published bytes exactly when, and only when, dropping that one newline gives the pinned hash.
for _name, _pin in RELEASED_SOURCE_PINS.items():
    _path = SOURCE_DIR / _name
    _raw = _path.read_bytes()
    if sha256_bytes(_raw) != _pin and _raw.endswith(b"\n") and sha256_bytes(_raw[:-1]) == _pin:
        _path.write_bytes(_raw[:-1])
    require(sha256_file(_path) == _pin, "Released module differs from its pin: " + _name)
COMPETITION_ROOT = None
for _root in ([CONFIG["INPUT_ROOT"] / "competitions/rsna-knee-abnormality-detection",
               CONFIG["INPUT_ROOT"] / "rsna-knee-abnormality-detection"]
              if CONFIG["COMPETITION_ROOT"] is None else [Path(CONFIG["COMPETITION_ROOT"])]):
    if all((_root / _name).is_file() for _name in ("test.csv", "sample_submission.csv", "test_series.csv")):
        COMPETITION_ROOT = _root
        break
require(COMPETITION_ROOT is not None, "Attach the authorized competition inputs or configure the actual mount")
TEST_IDS = pd.read_csv(COMPETITION_ROOT / "test.csv", dtype={"StudyInstanceUID": str})["StudyInstanceUID"].tolist()
SAMPLE_IDS = pd.read_csv(COMPETITION_ROOT / "sample_submission.csv", dtype={"StudyInstanceUID": str})["StudyInstanceUID"].tolist()
require(len(TEST_IDS) == len(set(TEST_IDS)) and TEST_IDS == SAMPLE_IDS, "Test and sample row order differ")
EXPECTED_COLUMNS = ["StudyInstanceUID", *LABELS]
INFERENCE = CONFIG["MODE"] == "inference"
require(not COMPETITION_RERUN or INFERENCE, "Competition reruns require full inference")
VISIBLE = set(TEST_IDS) == set(ORACLE["study_ids"])
STAGE_DIR = CONFIG["OUTPUT_ROOT"] / ("public_v35_stage_" + RUN_TAG)
STAGE_DIR.mkdir(parents=True, exist_ok=False)
ROOT_SUBMISSION = CONFIG["OUTPUT_ROOT"] / "submission.csv"
require(not ROOT_SUBMISSION.exists() and not ROOT_SUBMISSION.is_symlink(), "A root submission already exists; start a fresh run")
PREDICTION_NAME = "submission.csv" if INFERENCE else "canary_predictions.csv"


def find_input(kind, ref, *parts):
    owner, _, slug = ref.rpartition("/")
    for root in [CONFIG["INPUT_ROOT"] / kind / owner / slug, CONFIG["INPUT_ROOT"] / kind / slug, CONFIG["INPUT_ROOT"] / slug]:
        node = root.joinpath(*parts)
        if node.exists():
            return node
    raise FileNotFoundError(f"input not mounted: {kind} {ref} {'/'.join(parts)}")


def save_new(path, value):
    with Path(path).open("x") as stream:
        json.dump(value, stream, sort_keys=True, indent=2, allow_nan=False)
        stream.write("\n")


def rank_transform(values):
    # The reader's own transform: per column, argsort twice, divided by max(1, N - 1). Non-finite becomes 0.5.
    values = np.asarray(values, dtype=np.float64)
    ranks = np.argsort(np.argsort(values, axis=0), axis=0).astype(np.float64) / max(1, values.shape[0] - 1)
    ranks[~np.isfinite(ranks)] = 0.5
    return ranks


def owned_ok(owned):
    return (owned.get("status") == "PASS_OWNED_COMMAND" and owned.get("command_returncode") == 0 and
            owned.get("cleanup_on_success") is True and owned.get("leader_reaped") is True and
            owned.get("teardown", {}).get("unresolved") is False)


_name = "_owned_public_v35_watchdog"
require(_name not in sys.modules, "Owned watchdog identity occupied")
_spec = importlib.util.spec_from_file_location(_name, WATCHDOG_PATH)
_watchdog = importlib.util.module_from_spec(_spec)
sys.modules[_name] = _watchdog
_spec.loader.exec_module(_watchdog)
require(Path(_watchdog.__file__).resolve() == WATCHDOG_PATH.resolve(), "Owned watchdog origin changed")


def run_owned(command, absolute_deadline, log_name, cwd, env):
    remaining = absolute_deadline - time.monotonic()
    require(remaining > 65, "Insufficient owned budget left for " + log_name)
    with (STAGE_DIR / log_name).open("x") as log:
        return _watchdog.run_owned(command, timeout=remaining + 2.0, absolute_deadline=absolute_deadline + 2.0,
                                   cleanup_on_success=True, cwd=str(cwd), env=env or os.environ.copy(), stdout=log,
                                   stderr=subprocess.STDOUT)


# Arm 1, the mirror arm: the reader's own inference path, unchanged. Its outputs stay in the stage directory.
MIRROR_DIR = STAGE_DIR / "mirror"
try:
    _owned_mirror = run_owned([sys.executable, "-u", str(READER_PATH), "--input-root", str(CONFIG["INPUT_ROOT"]),
                               "--competition-root", str(COMPETITION_ROOT), "--output-dir", str(MIRROR_DIR),
                               "--mode", CONFIG["MODE"], "--max-studies", str(CONFIG["MAX_CANARY_STUDIES"]),
                               "--prep-workers", str(CONFIG["PREP_WORKERS"])],
                              FULL_DEADLINE, "mirror_reader.log", SOURCE_DIR, None)
    require(owned_ok(_owned_mirror), "Mirror arm lifecycle incomplete; no submission is exported")
    _receipt = json.loads((MIRROR_DIR / "native_inference_receipt.json").read_bytes())
    mirror_bytes = (MIRROR_DIR / PREDICTION_NAME).read_bytes()
    require(_receipt.get("native_inference_completed") is True and _receipt.get("mode") == CONFIG["MODE"] and
            _receipt.get("source_sha256") == READER_SHA256 and _receipt.get("checkpoint_sha256") == CHECKPOINT_SHA256 and
            _receipt.get("competition_submitted") is False and _receipt.get("official_score") is None and
            _receipt.get("prediction_file_sha256") == sha256_bytes(mirror_bytes), "Mirror receipt drift")
    if INFERENCE:
        require(_receipt.get("full_test_coverage") is True and _receipt.get("studies") == len(TEST_IDS), "Incomplete full-test coverage")
    expected_rows = len(TEST_IDS) if INFERENCE else min(len(TEST_IDS), CONFIG["MAX_CANARY_STUDIES"])
    mirror_frame = pd.read_csv(io.BytesIO(mirror_bytes), dtype={"StudyInstanceUID": str})
    require(mirror_frame.columns.tolist() == EXPECTED_COLUMNS and mirror_frame["StudyInstanceUID"].tolist() == TEST_IDS[:expected_rows],
            "Mirror schema, coverage or row order drift")
    mirror_values = np.load(MIRROR_DIR / "probabilities.npz")["values"].astype(np.float64)
    require(mirror_values.shape == (expected_rows, 12) and np.isfinite(mirror_values).all(), "Mirror probabilities drift")
    ranks_mirror = rank_transform(mirror_values)
    require(np.array_equal(ranks_mirror.astype(np.float32), mirror_frame[LABELS].to_numpy(np.float32)),
            "Recomputed mirror ranks differ from the mirror CSV")
    if INFERENCE and VISIBLE:
        require(sha256_bytes(mirror_bytes) == VISIBLE_MIRROR_SHA256, "Visible mirror CSV differs from the reference")
except Exception as _error:
    save_new(STAGE_DIR / "public_v35_failure.json", dict(status="FAIL_MIRROR_ARM", error_type=type(_error).__name__,
                                                        message=str(_error)[:500], competition_submitted=False, official_score=None))
    raise
(STAGE_DIR / "mirror_only.csv").write_bytes(mirror_bytes)

# Arm 2, the goodpjw ConvNeXt arm: inference only. Any arm-level failure leaves the mirror-only submission.
ARM = dict(name="goodpjw2008 ConvNeXt 2.5D V1", weight=CONFIG["CONVNEXT_WEIGHT"], status=None, reason=None,
           gate_threshold_s=CONVNEXT_THRESHOLD_S, whole_budget_s=WHOLE_BUDGET)
convnext_values = None
arm_ok = False
if not INFERENCE:
    ARM.update(status="NOT_RUN_CANARY", reason="canary mode runs the mirror arm only")
else:
    _remaining_before_arm = FULL_DEADLINE - time.monotonic()
    ARM.update(remaining_before_arm_s=_remaining_before_arm)
    if _remaining_before_arm < CONVNEXT_THRESHOLD_S:
        ARM.update(status="SKIPPED_TIME_GATE",
                   reason="remaining %.0f s is below the gate threshold %.0f s" % (_remaining_before_arm, CONVNEXT_THRESHOLD_S))
    else:
        try:
            ASSET_DIR = find_input("datasets", CONVNEXT_DATASET)
            asset_paths = {}
            for _name in CHECKPOINT_NAMES:
                _path = ASSET_DIR / _name
                _pin = ASSET_PINS[_name]
                require(_path.is_file() and _path.stat().st_size == _pin["bytes"] and sha256_file(_path) == _pin["sha256"],
                        "Released checkpoint differs from its pin: " + _name)
                asset_paths[_name] = _path
            CONV_SITE = SOURCE_DIR / "decoder_site"
            CONV_SITE.mkdir()
            # Best effort, one wheel at a time: a wheel that cannot install is recorded and never fails the arm.
            DECODER_RESULTS = []
            for _name in DECODER_NAMES:
                _entry = dict(wheel=_name, installed=False, returncode=None, reason=None, stderr_tail=None)
                try:
                    _path = ASSET_DIR / _name
                    _pin = ASSET_PINS.get(_name)
                    require(_pin is not None and _path.is_file() and _path.stat().st_size == _pin["bytes"] and
                            sha256_file(_path) == _pin["sha256"], "wheel missing or differs from its pin")
                    _log_name = "pip_" + str(len(DECODER_RESULTS)) + ".log"
                    _owned_pip = run_owned(PIP_COMMAND + ["install", "--quiet", "--no-index", "--no-deps", "--target",
                                                          str(CONV_SITE), str(_path)],
                                           FULL_DEADLINE - CONFIG["CONVNEXT_RESERVE_S"], _log_name, SOURCE_DIR, None)
                    _installed = owned_ok(_owned_pip)
                    _entry.update(installed=_installed, returncode=_owned_pip.get("command_returncode"),
                                  stderr_tail=None if _installed else (STAGE_DIR / _log_name).read_text()[-300:])
                except Exception as _error:
                    _entry["reason"] = type(_error).__name__ + ": " + str(_error)[:200]
                DECODER_RESULTS.append(_entry)
            ARM["decoder_install"] = DECODER_RESULTS
            ARM["python"] = sys.version
            CHECKPOINT_PATHS = [str(asset_paths[_n]) for _n in CHECKPOINT_NAMES]
            _config = dict(study_ids=TEST_IDS, labels=LABELS, competition_dir=str(COMPETITION_ROOT), checkpoints=CHECKPOINT_PATHS,
                           device=CONFIG["CONVNEXT_DEVICE"], workers=CONFIG["CONVNEXT_WORKERS"], source_dir=str(SOURCE_DIR),
                           output_dir=str(STAGE_DIR / "convnext"),
                           file_pins=[dict(path=p, bytes=ASSET_PINS[Path(p).name]["bytes"], sha256=ASSET_PINS[Path(p).name]["sha256"])
                                      for p in CHECKPOINT_PATHS] +
                                     [dict(path=str(SOURCE_DIR / _n), bytes=(SOURCE_DIR / _n).stat().st_size, sha256=_s)
                                      for _n, _s in RELEASED_SOURCE_PINS.items()],
                           oracle=(dict(ORACLE, tolerance=0.0002) if VISIBLE else None))
            with (SOURCE_DIR / "convnext_config.json").open("x") as _stream:
                json.dump(_config, _stream, indent=2, allow_nan=False)
            _child_env = dict(os.environ, PYTHONPATH=os.pathsep.join([str(SOURCE_DIR), str(CONV_SITE), os.environ.get("PYTHONPATH", "")]))
            _owned_convnext = run_owned([sys.executable, "-u", str(CHILD_PATH), "--config", str(SOURCE_DIR / "convnext_config.json")],
                                        FULL_DEADLINE - CONFIG["CONVNEXT_RESERVE_S"], "convnext_worker.log", SOURCE_DIR, _child_env)
            require(owned_ok(_owned_convnext), "ConvNeXt lifecycle incomplete")
            _child_receipt = json.loads((STAGE_DIR / "convnext" / "receipt.json").read_bytes())
            require(_child_receipt.get("status") == "PASS_CONVNEXT_ARM", "ConvNeXt receipt status differs")
            _child_arrays = np.load(STAGE_DIR / "convnext" / "convnext_probabilities.npz")
            require(_child_arrays["study_uids"].tolist() == TEST_IDS, "ConvNeXt study order drift")
            convnext_values = _child_arrays["values"].astype(np.float64)
            _child_status = _child_arrays["status"]
            require(convnext_values.shape == (len(TEST_IDS), 12) and np.isfinite(convnext_values).all() and
                    ((convnext_values >= 0) & (convnext_values <= 1)).all(), "ConvNeXt probabilities drift")
            require((_child_status == 0).mean() >= CONFIG["MIN_CONVNEXT_OK_FRACTION"], "Too many ConvNeXt study failures")
            if VISIBLE:
                require(_child_receipt["oracle"] is not None and _child_receipt["oracle"]["passed"],
                        "ConvNeXt visible comparison with the author's probabilities failed")
            arm_ok = True
            ARM.update(status="OK", studies=len(TEST_IDS), ok_studies=int((_child_status == 0).sum()),
                       input_failed=int((_child_status == 1).sum()), model_failed=int((_child_status == 2).sum()),
                       decode=_child_receipt.get("decode"), probe=_child_receipt.get("probe"),
                       oracle=_child_receipt.get("oracle"), elapsed_seconds=_child_receipt.get("elapsed_seconds"),
                       device_name=_child_receipt.get("device_name"), checkpoint_sha256=_child_receipt.get("checkpoint_sha256"))
        except Exception as _error:
            convnext_values = None
            arm_ok = False
            ARM.update(status="FAILED_MIRROR_ONLY", reason=type(_error).__name__ + ": " + str(_error)[:300])

# Blend (predeclared): S = 0.70 * rank(mirror) + 0.30 * rank(ConvNeXt), final = rank(S). Written before the export.
if INFERENCE and arm_ok:
    ranks_convnext = rank_transform(convnext_values)
    score = CONFIG["MIRROR_WEIGHT"] * ranks_mirror + CONFIG["CONVNEXT_WEIGHT"] * ranks_convnext
    final_ranks = rank_transform(score)
    frame_blend = pd.DataFrame(final_ranks.astype(np.float32), columns=LABELS)
    frame_blend.insert(0, "StudyInstanceUID", TEST_IDS)
    with (STAGE_DIR / "blended.csv").open("x") as _stream:
        frame_blend.to_csv(_stream, index=False)
    frame_convnext = pd.DataFrame(ranks_convnext.astype(np.float32), columns=LABELS)
    frame_convnext.insert(0, "StudyInstanceUID", TEST_IDS)
    with (STAGE_DIR / "convnext_only.csv").open("x") as _stream:
        frame_convnext.to_csv(_stream, index=False)
    np.savez_compressed(STAGE_DIR / "arm_probabilities.npz", study_uids=np.asarray(TEST_IDS, dtype=str), labels=np.asarray(LABELS),
                        mirror_values=mirror_values, convnext_values=convnext_values, ranks_mirror=ranks_mirror,
                        ranks_convnext=ranks_convnext, blend_score=score, blended_ranks=final_ranks)
    FINAL_BYTES = (STAGE_DIR / "blended.csv").read_bytes()
    FINAL_SOURCE = "blended"
else:
    FINAL_BYTES = mirror_bytes
    FINAL_SOURCE = "mirror_only" if INFERENCE else "canary_not_exported"
run_receipt = dict(status="PASS_V35_" + FINAL_SOURCE.upper(), final_source=FINAL_SOURCE,
                   final_csv_sha256=sha256_bytes(FINAL_BYTES), mirror_csv_sha256=sha256_bytes(mirror_bytes),
                   blend=dict(mirror_weight=CONFIG["MIRROR_WEIGHT"], convnext_weight=CONFIG["CONVNEXT_WEIGHT"], applied=arm_ok,
                              transform="argsort.argsort / max(1, N - 1), non-finite -> 0.5"),
                   convnext_arm=ARM, remaining_budget_s=FULL_DEADLINE - time.monotonic(), whole_budget_s=WHOLE_BUDGET,
                   competition_rerun=COMPETITION_RERUN, studies=len(TEST_IDS), competition_submitted=False, official_score=None)
save_new(STAGE_DIR / "public_v35_run_receipt.json", run_receipt)
if INFERENCE:
    # The last write of the notebook: the root submission, exclusive, after every check above.
    with ROOT_SUBMISSION.open("xb") as _export:
        _export.write(FINAL_BYTES)
    require(sha256_file(ROOT_SUBMISSION) == sha256_bytes(FINAL_BYTES), "Root export differs from the validated file")
print({"status": run_receipt["status"], "final_source": FINAL_SOURCE, "studies": len(TEST_IDS), "convnext_arm": ARM["status"],
       "root_submission_exported": INFERENCE, "competition_submitted": False, "official_score": None})


## 7. Verify the run without displaying study identifiers

A successful run is not sufficient by itself. The launcher has already checked the sample columns and row order, finite values in [0, 1], full coverage in inference mode, the file digests in the receipts, and the blend arithmetic. The cell below re-reads the exported file and prints only aggregate counts, hashes and arm statuses.

The stage directory contains the mirror and ConvNeXt outputs, the receipts and the logs. Some carry study identifiers. Use them only in your authorized workspace; do not publish them as demonstration data or raw notebook output. A canary file must never be submitted as a full-test prediction.


In [ ]:
# Aggregate verification only: no study identifiers or prediction values are printed.
if INFERENCE:
    _root_bytes = ROOT_SUBMISSION.read_bytes()
    _receipt = json.loads((STAGE_DIR / "public_v35_run_receipt.json").read_bytes())
    require(sha256_bytes(_root_bytes) == _receipt["final_csv_sha256"], "Root file drift")
    _arm = _receipt["convnext_arm"]
    print({"root_submission_sha256": sha256_bytes(_root_bytes), "bytes": len(_root_bytes),
           "final_source": _receipt["final_source"], "convnext_arm": _arm["status"], "studies": _receipt["studies"],
           "decode_failed_studies": (_arm.get("decode") or {}).get("decode_failed_studies"),
           "competition_submitted": False, "official_score": None})
else:
    print({"canary": True, "root_submission_exported": False, "official_score": None})


## What to change in a useful fork

Begin with this exact baseline and save its receipt. For a new experiment, change one mechanism, write down the hypothesis before running, and compare against the same baseline on a suitable independent validation split.

| Question | Small, interpretable experiment | Main caution |
|---|---|---|
| Does the mirrored view contribute independent information? | Compare plain, mirror and their equal probability mean | Keep the checkpoint, studies and preprocessing identical |
| Is acquisition geometry responsible for the gain? | Compare the documented plane-aware mirror with an explicitly defined alternative | Channel statistics and reflection order must stay explicit |
| Do two trained readers add diversity? | Test a preregistered blend with an independently trained, permitted checkpoint | Shared training data or ancestry can reduce diversity |
| Can preparation use fewer CPU threads? | Change `PREP_WORKERS` and measure elapsed time/peak memory | A source check does not measure speed |

Do not tune weights repeatedly on a reused exposed panel or infer causality from a single public leaderboard change. The publisher's Gold58 set was reused for selection, so it is not independent cross-validation. The older plain-versus-affine experiment belongs to a separate training question; it is not evidence that this mirror recipe improved or failed.


## Credits, changes and required acknowledgement

## Upstream lineage

- **nartaa / Danial Zakaria**: selected learned checkpoint and plain/anatomical-mirror inference recipe in [RSNA Knee 0949 Anatomical Mirror, V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), Apache-2.0. The displayed 0.949 score belongs to that upstream notebook.
- **dreaddevelopment**: [Raptor CoAtNet/MIL recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). The selected nartaa weights are the publisher's retrained checkpoints, not these original baseline weights.
- **goodpjw2008**: the three-fold ConvNeXt 2.5D reader and its released sources (`preprocess.py`, `knee.py`, `infer.py`), Apache-2.0, dataset [goodpjw2008 / rsna-knee-2-5d-convnext-reader](https://www.kaggle.com/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader), version 1. The sources are embedded byte for byte and checked against their published hashes.
- **timm / Ross Wightman and contributors**: [CoAtNet implementation and pretrained component lineage](https://github.com/huggingface/pytorch-image-models). Third-party components retain their own licenses.
- The checkpoint publisher credits weak-label contributions from [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), together with the publisher's report-derived labels. These tables and source reports are not included.
- **RSNA and competition data contributors**: data use remains subject to the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

## Original implementation and changes

The ConvNeXt glue, the blend, the time gate, the decoder probe and this orchestration are original to this notebook and are released under MIT, copyright 2026 cubres. The owned-process watchdog (`owned_watchdog.py`) is Apache-2.0 original code.

`public_mirror_reader.py` (version 34, sha256 `e9ec648e…`) is an original readable implementation of the visible upstream recipe, not a copy of private campaign code or an independently trained model. Relative to the strict reader of earlier versions, it adopts the public reference's fault semantics: unreadable headers keep a neutral record, unreadable slices are skipped, unbuildable studies are predicted from zero inputs, failed model calls give 0.5, and no study is dropped. It adds checkpoint and source identity constants, schema and coverage gates, tensor-only deserialization, bounded thread queues without multiprocessing, separate canary output names and create-only outputs. It does not execute the upstream fast-reader payload. The owned-process watchdog (`owned_watchdog.py`, sha256 `54636411…`) is original Apache-2.0 code. The selected checkpoint bytes, geometry and aggregation formulas remain attributed to the public references. These differences require native validation; this version makes no reproduced-score or bit-equivalence claim.

## Publisher's required OAI / NDA acknowledgement

The following paragraph is preserved verbatim from the upstream publisher's credits and dataset description. It acknowledges the upstream research lineage; distributing this reader does not assert that this repository obtained or redistributed controlled-access OAI records.

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study 3407](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

The upstream weights package grants only its stated model-use permission. OAI access and redistribution remain governed by NDA. This notebook contains no OAI or competition MRI, reports, source label tables, participant identifiers or learned checkpoint payloads.


### Source and model licenses stay separate

- Active original reader and upstream source lineage: **Apache-2.0**. [Complete Apache license](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/Apache-2.0.txt).
- Newly written notebook explanations/configuration/diagrams: **MIT**, copyright 2026 cubres. [MIT text](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/MIT.txt).
- Selected checkpoint: publisher's **Other / research and educational model-use terms**, linked as original V1; no checkpoint redistribution or sublicensing grant is assumed.
- Historical inactive source below retains all original notices, including its DINOv3 license. This refresh does not relicense that archive.

The publisher's model-use statement is:

> These learned weights are provided for research and educational use, including the RSNA Knee competition. This model-use permission grants no rights to obtain, publish, redistribute or re-identify OAI participant data, source MRI images, reports or other controlled-access records. Access to OAI data remains governed by NDA's terms. Third-party source code and pretrained components retain their respective licences. No CC0 or other open-data licence is applied to OAI data or raw corpora by this model release.

[Versioned model-use explanation](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/MODEL_USAGE.md) · [License boundaries](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The version-34 reader is the cell in section 3. No weights, source MRI, reports, label tables or participant records are embedded in this notebook source.


<details><summary>Complete Apache-2.0 license for the active reader</summary>

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "{}"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2019 Ross Wightman

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
</details>


## Preserved history

Every Version 34 cell appears below as an inactive cell, in its original order. Code cells are raw, so nothing runs; their sources are verbatim, and their metadata holds only small keys (cell type, original index, original identifier and source hashes). The earlier Version 33 source is preserved inside Version 34 in the same way. Nothing in the preserved history is duplicated in metadata.


# The bee's knees: reading MRI with an anatomical mirror

![An artistic knee joint with mirrored MRI planes](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/knee-anatomical-mirror-hero-v1.png)

*AI-generated artistic illustration, not measured MRI. [Prompt and artwork provenance](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/ARTWORK_PROVENANCE.json).*

A knee MRI is a collection of acquisitions, not one picture. This notebook shows how a strong public method turns that collection into a finding vector: choose the slices, preserve their physical geometry, read adjacent triplets, pool the evidence, and average a plain and an anatomically mirrored view. The implementation is deliberately readable enough to fork.

The checkpoint and mirror recipe come from [nartaa / Danial Zakaria's scored V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), building on dreaddevelopment's Raptor and timm's CoAtNet. **The upstream notebook displayed 0.949. That is nartaa's score, not ours.**

## Results so far

| Entry | Score | Basis |
|---|---|---|
| nartaa V2: upstream recipe and checkpoint | 0.949 | Their score on their notebook. Attributed here, not reproduced |
| This public notebook (card) | 0.941 | Official rows 56759401 (V31), 56779708 and 56779710 (V32), all 0.941 |
| Account's verified best | 0.943 | Private notebook, row 56859861 (V8) |
| Public frontier, observed 2026-10-08 | Six notebooks at 0.950 (blends) | Public leaderboard. Not ours |
| Private V14 | pending | Row 56948803; no score |
| This version, V34 | pending | No score is claimed until a completed official submission exists |

Earlier versions and their displayed results are recorded in the preserved history at the end. Version 34 changes the active code path. Its own native GPU run and official submission are still to come.

## What changed in version 34

Version 34 starts its clock in the first statement of the notebook, before any import, staging or CUDA work, so the whole time budget includes setup. Inference runs as the leader of its own process group under the same owned-process watchdog our private campaign uses. The watchdog cleans up the whole group on success as well as on failure. The output is staged and checked for coverage, schema, row order, finiteness, source and checkpoint identity, and file digest. Only then is `/kaggle/working/submission.csv` written, with exclusive creation.

Version 34 also replaces the strict reader with a tolerant one. Section 3 explains why that matters.

[Scientific contract](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/SCIENTIFIC_CONTRACT.md) · [Licenses and model-use terms](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The active reader source is the cell in section 3. Its sha256 is pinned in the launcher.


## Read the experiment before running it

![MRI volume, windows, two views, attention and ranks](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/anatomical_mirror_pipeline_v2.png)

| Choice | This baseline | Why it matters |
|---|---|---|
| Checkpoint | One selected public SWA checkpoint, exact V1 hash | Reproducible identity; this is not a new independent model |
| Physical field of view | 140 mm center crop | Pixels alone do not fix the anatomical field of view |
| Source volume | 96 slices at 384 × 384 | Preserve the pretrained geometry and five acquisition slots |
| Model input | 94 adjacent triplets, center crop to 320 × 320 | Keep neighboring-slice information without resizing again |
| Mirror | Acquisition channels for sagittal centers; image width otherwise | A reflection must follow the acquisition plane |
| Pooling | Finding-specific attention over windows | Each finding can attend to different evidence |
| Combine views | Mean sigmoid probabilities, then per-finding ordinal ranks | This is not logit averaging or clinical calibration |

There is no training step and no model download. Attach [the publisher's original weights dataset V1](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007/versions/1), use competition inputs you are authorized to access, and choose one or more **T4 GPUs**. The qualified environment used two. The qualified container has Torch `2.11.0+cu128` and timm `1.0.29`; library checks fail before any model is loaded.

The baseline runs all current test rows in `inference` mode. Set `MODE="canary"` for a three-study smoke test. That writes a separately named file and is not an accuracy measurement. A notebook run only creates predictions. It does not submit to the competition.


## 1. A small configuration you can actually use

The first code cell starts the clock. Its first statement reads the monotonic time before any import, staging or CUDA work, so the budget covers setup as well as inference. A competition rerun (`KAGGLE_IS_COMPETITION_RERUN` set to true or 1) uses the published 32,400-second GPU limit. A commit uses the 1,800-second notebook limit. Each budget keeps a 50-second margin, so the whole-run deadlines are 32,350 s and 1,750 s. An unknown flag value stops the run instead of guessing.

Change `MODE` and `PREP_WORKERS` for a bounded smoke test or a preparation-speed experiment. Change paths only if your attached data layout differs. The baseline geometry and equal view weights live in the reader in section 3. If you change them, update `READER_SHA256` and document the change before treating a fork as reproducible.


## 2. Respect the acquisition geometry

| Slot | Source slice indices | Count |
|---|---|---:|
| Sagittal, fluid-sensitive | 0–25 | 26 |
| Sagittal, non-fluid-sensitive | 26–47 | 22 |
| Coronal, fluid-sensitive | 48–65 | 18 |
| Coronal, non-fluid-sensitive | 66–77 | 12 |
| Axial, any sequence | 78–95 | 18 |

We prefer the first suitable unused series in metadata order, falling back within the requested plane. DICOM position and orientation order the slices, with `InstanceNumber` as a fallback. Apply the modality LUT and MONOCHROME1 inversion before robust per-series intensity scaling. Sample the 2–98% index span, use the selected images' 2nd/98th pixel percentiles, crop a 140 mm field of view, then area-resize and quantize to uint8.

The volume layout survives missing acquisition slots. A study with no usable acquisition fails instead of receiving model priors. Window selection preserves the upstream span and can cross slot boundaries; the center determines the mirror operation.

![Acquisition-channel reversal and width reflection before normalization](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/mirror_before_normalization_v2.png)

Mirror the **uint8** triplet first. Then crop the central 320 × 320 region (32 pixels from each edge of 384 × 384), and only then apply the three channel-specific ImageNet means and standard deviations. Reversing channels after normalization would alter the recipe because those statistics differ.


## 3. The reader, and why tolerance matters

The next cell writes the exact version-34 reader (sha256 `e9ec648e…`). The launcher verifies those bytes before it starts. The same tolerant reader, its diff from the strict version, the offline test harness and the visible-run receipts are published in the [official-results folder](https://github.com/cubres/rsna-knee-abnormality-detection/tree/b8e1d6e89c391e27821a41455d5f42336f70420d/official-results-2026-10-08) of the [repository](https://github.com/cubres/rsna-knee-abnormality-detection). The architecture and the happy-path arithmetic are the same as before. What changed is how the reader treats bad input.

An earlier private version of this pipeline ran the same architecture under a strict reader. That reader stopped the whole job on an unreadable DICOM header, an unreadable or non-2D pixel array, a study with no usable slice, any per-study model error, or a device count other than two. Its visible three-study run passed. Its rerun on the hidden test cohort, about 1,300 studies, ended with an unhandled error and no score. We cannot see the hidden log, so this is our most likely explanation rather than a proven one. A cohort that large is very likely to contain at least one file that a strict reader rejects, and one such file should not zero the submission.

The public reference handles these cases deliberately. An unreadable header keeps its file as a neutral record. An unreadable slice is skipped and its slot stays zero. A study whose volume cannot be built is predicted from a zero volume and mask. A failed model call gives that study 0.5 probabilities. Every study still receives a prediction. In offline tests with synthetic DICOM, this reader matched the reference on truncated, garbage, multi-frame, NaN and missing-series studies. Its output was identical to the strict reader's on healthy studies. Those tests do not measure accuracy, and the GPU path still needs its own native run.

The general lesson is that a competition reader is part of the scoring pipeline. Its failure policy should be stated, tested on bad inputs, and chosen so that every study stays in the output. The functions to read are `build_volume` (the fault policy), `prepare_windows` and `predict_pair` (the model path), and `run` (coverage and ranks).


## 4. The owned process group

A timeout on one child is not enough. A reader that starts workers can leave descendants behind. A notebook that exports a file before those processes are cleaned up can hide a problem. The next cell writes the owned-process watchdog (sha256 `54636411…`), the same module our private campaign uses. It runs the reader as the leader of a fresh process group, enforces an absolute deadline, and on success terminates the group and verifies that teardown has resolved before the launcher accepts any output. It is Apache-2.0, like the reader.


## 4. Let each finding choose its windows

CoAtNet encodes the triplets in chunks of 16. LayerNorm and a small attention head give each finding its own softmax over the windows. The weighted sum of window features produces one vector per finding; a finding-specific classifier turns it into a logit. Dropout is disabled in evaluation mode.

```text
attention[k, c] = softmax over k (attention_head(feature[k]))
pooled[c]       = sum over k (attention[k, c] * feature[k])
logit[c]        = classifier[c](pooled[c])
probability[c]  = (sigmoid(plain_logit[c]) + sigmoid(mirror_logit[c])) / 2
```

The final CSV uses the upstream per-column `argsort().argsort() / max(1, N-1)` transform. These ordinal ranks are useful for ranking; they are not calibrated probabilities. Preserve the source's NumPy tie behavior. Canary ranks depend on the canary's N and therefore cannot establish full-test prediction equivalence.


## 5. Run in stages and export last

The launcher checks the pinned runtime, the reader and watchdog bytes, and the competition mount before any checkpoint is loaded. It starts the reader under the owned watchdog with the absolute deadline set in the first cell. The reader writes to a fresh stage directory, `/kaggle/working/public_v34_stage_<run>/`, and never to the submission path. After the child's process group is cleaned up and its receipt is read, the staged file is validated: sample columns and row order, finite values in [0, 1], full coverage, and the SHA-256 bound in the native receipt. Only then is `/kaggle/working/submission.csv` written, with exclusive creation. If any check fails, no submission file appears.

The checkpoint requires the publisher's weights dataset attachment. Source SHA-256 and the checkpoint digest are distinct pins. The reader hashes the actual checkpoint file before deserialization. No code license grants rights to redistribute checkpoint bytes or controlled-access records.


## 6. Verify the file without displaying study identifiers

A successful run is not sufficient by itself. The launcher has already checked the sample columns and row order, finite values in [0, 1], full coverage in inference mode, and the file digest bound into the native receipt. The cell below re-reads the exported file and prints only aggregate counts and hashes.

The staged directory contains `probabilities.npz`, the reader's log and the receipt. Some of these carry study identifiers. Use them only in your authorized workspace. Do not publish them as demonstration data or as raw notebook output. A canary file must never be submitted as a full-test prediction.


## What to change in a useful fork

Begin with this exact baseline and save its receipt. For a new experiment, change one mechanism, write down the hypothesis before running, and compare against the same baseline on a suitable independent validation split.

| Question | Small, interpretable experiment | Main caution |
|---|---|---|
| Does the mirrored view contribute independent information? | Compare plain, mirror and their equal probability mean | Keep the checkpoint, studies and preprocessing identical |
| Is acquisition geometry responsible for the gain? | Compare the documented plane-aware mirror with an explicitly defined alternative | Channel statistics and reflection order must stay explicit |
| Do two trained readers add diversity? | Test a preregistered blend with an independently trained, permitted checkpoint | Shared training data or ancestry can reduce diversity |
| Can preparation use fewer CPU threads? | Change `PREP_WORKERS` and measure elapsed time/peak memory | A source check does not measure speed |

Do not tune weights repeatedly on a reused exposed panel or infer causality from a single public leaderboard change. The publisher's Gold58 set was reused for selection, so it is not independent cross-validation. The older plain-versus-affine experiment belongs to a separate training question; it is not evidence that this mirror recipe improved or failed.


## Credits, changes and required acknowledgement

## Upstream lineage

- **nartaa / Danial Zakaria**: selected learned checkpoint and plain/anatomical-mirror inference recipe in [RSNA Knee 0949 Anatomical Mirror, V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), Apache-2.0. The displayed 0.949 score belongs to that upstream notebook.
- **dreaddevelopment**: [Raptor CoAtNet/MIL recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). The selected nartaa weights are the publisher's retrained checkpoints, not these original baseline weights.
- **timm / Ross Wightman and contributors**: [CoAtNet implementation and pretrained component lineage](https://github.com/huggingface/pytorch-image-models). Third-party components retain their own licenses.
- The checkpoint publisher credits weak-label contributions from [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), together with the publisher's report-derived labels. These tables and source reports are not included.
- **RSNA and competition data contributors**: data use remains subject to the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

## Original implementation and changes

`public_mirror_reader.py` (version 34, sha256 `e9ec648e…`) is an original readable implementation of the visible upstream recipe, not a copy of private campaign code or an independently trained model. Relative to the strict reader of earlier versions, it adopts the public reference's fault semantics: unreadable headers keep a neutral record, unreadable slices are skipped, unbuildable studies are predicted from zero inputs, failed model calls give 0.5, and no study is dropped. It adds checkpoint and source identity constants, schema and coverage gates, tensor-only deserialization, bounded thread queues without multiprocessing, separate canary output names and create-only outputs. It does not execute the upstream fast-reader payload. The owned-process watchdog (`owned_watchdog.py`, sha256 `54636411…`) is original Apache-2.0 code. The selected checkpoint bytes, geometry and aggregation formulas remain attributed to the public references. These differences require native validation; this version makes no reproduced-score or bit-equivalence claim.

## Publisher's required OAI / NDA acknowledgement

The following paragraph is preserved verbatim from the upstream publisher's credits and dataset description. It acknowledges the upstream research lineage; distributing this reader does not assert that this repository obtained or redistributed controlled-access OAI records.

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study 3407](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

The upstream weights package grants only its stated model-use permission. OAI access and redistribution remain governed by NDA. This notebook contains no OAI or competition MRI, reports, source label tables, participant identifiers or learned checkpoint payloads.


### Source and model licenses stay separate

- Active original reader and upstream source lineage: **Apache-2.0**. [Complete Apache license](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/Apache-2.0.txt).
- Newly written notebook explanations/configuration/diagrams: **MIT**, copyright 2026 cubres. [MIT text](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/MIT.txt).
- Selected checkpoint: publisher's **Other / research and educational model-use terms**, linked as original V1; no checkpoint redistribution or sublicensing grant is assumed.
- Historical inactive source below retains all original notices, including its DINOv3 license. This refresh does not relicense that archive.

The publisher's model-use statement is:

> These learned weights are provided for research and educational use, including the RSNA Knee competition. This model-use permission grants no rights to obtain, publish, redistribute or re-identify OAI participant data, source MRI images, reports or other controlled-access records. Access to OAI data remains governed by NDA's terms. Third-party source code and pretrained components retain their respective licences. No CC0 or other open-data licence is applied to OAI data or raw corpora by this model release.

[Versioned model-use explanation](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/MODEL_USAGE.md) · [License boundaries](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The version-34 reader is the cell in section 3. No weights, source MRI, reports, label tables or participant records are embedded in this notebook source.


<details><summary>Complete Apache-2.0 license for the active reader</summary>

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "{}"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2019 Ross Wightman

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
</details>


## Preserved historical source

Every original Version 33 cell remains below as an inactive raw cell with its original cell identifier. The exact original notebook bytes (203,753 bytes, SHA-256 `df02bfdc…`) are also preserved as a SHA-256-bound gzip/base64 snapshot in the notebook metadata, so the original JSON, outputs, cell types and metadata can be reconstructed exactly. The historical launcher and reader in those cells do not run in version 34.
